# B32-M + Manh's production vLLM server

B32-M (`dsv41-tools-submit-b32m.ipynb`) with the vLLM server started by **Manh's cell 2m** (his production command,
`dsv41-vllm-prod.md`), not by the solver. B32's solver and harness only call it: `http://127.0.0.1:18084/v1`, model `dsv41`.

Changed cells (everything else is B32-M as is):
- **Cell 00** settings: `ARC_PORT=18084`, `ARC_MAX_SERVER_RESTARTS=3` (his supervisor's restarts, B32's 3); B32-M's
  `ARC_WORKERS=80` and `ARC_CONTEXT_LIMIT=163840` stay (his server: `--concurrency 80`, `--context 163840`); the
  solver's server settings are gone.
- **Cell 0b** overlay: B32's pinned files with the solver + S25 `ARC_EXTERNAL_SERVER` + S26
  `ARC_EXTERNAL_SERVER_STATUS` (`pinned_candb_ext/`, branch `havu-s24-supervised-server`); `_MT_EXPECT` is B32-M's.
- **Manh's cells 1-5** as is: they install the newest vllm wheel of his `vllm-wheels` dataset (attach its latest
  version; cell 2 prints and records `vllm.__version__`).
- **Cell 2r** (B32-M's Engram row-chunk patch for the g5e1bccd07c wheel) is gone: his wheels from a6c3ee8c2b on load
  the Kaggle copy themselves. Cell 2's record of it and cell 4b's check of it went with it.
- **Cell 2m** (was "Manh's extra flags: OFF"): starts his server under his restart supervisor (`dsv41_vllm.py
  --supervise 3`: frees the port first, restarts the server after an exit, not past `ARC_STOP_NEW_S`), waits until
  ready, defines `MANH_BASE_URL` / `MANH_PORT` / `MANH_SERVER_LOG` / `MANH_SERVER_STATUS`. An arc2-2026-src without
  `--supervise` still works: one start, no restart (a warning says so).
- **Cell 3** solver: `ARC_EXTERNAL_SERVER=MANH_BASE_URL`, `ARC_EXTERNAL_SERVER_LOG`, `ARC_EXTERNAL_SERVER_STATUS`
  (S26: waits for each restart, re-queues the attempts a death cut), `ARC_MAX_SERVER_RESTARTS`, `ARC_MODEL_DIR` = his
  server's model, no staging wait.
- **Cell 4** final check: stops his supervisor, then his server.
- **Cell 4b** smoke report: reads the external server's lines (`server up after Ns (external, start K)`) and log,
  and prints the supervisor's starts.

To call another server from B32's solver: set `ARC_EXTERNAL_SERVER=http://host:port/v1` (served model name `dsv41`);
for one under Manh's supervisor also `ARC_EXTERNAL_SERVER_STATUS=<its supervisor.json>`.

In [ ]:
# B32-M fix (build_img_manh.py --text, 2026-10-29): Manh's notebook + B32's text harness on the E128 text
# model, with a restart supervisor (cell 2s), memory margins and the commit-report checks.
# B32-M + MANH'S SERVER (2026-10-30): B32-M with the vLLM server started by Manh's cell 2m (his production
# command, dsv41-vllm-prod.md) instead of the solver; the solver only calls it (port 18084, model dsv41).
# B32-M (2026-10-28): B32 with Manh's vLLM environment (build_b32m.py): his image, CUDA 13 and wheels; the
# E062 L4 replica's server config (DCP4, offload 32 GiB, graphs/seqs 96); 80 workers, KV gate 0.85.
# Cell 00 (E046 frozen-input candidate): submit Version 5's settings (Humming dense, CUDA graphs <= 64, max-num-seqs
# 64, util 0.945; Kaggle KV 1.55M) + one rollout per task for tasks with several test inputs (ARC_MULTI_TEST) with a
# (1 + 0.5 (n - 1)) x output budget / turn cap (ARC_MULTI_BUDGET); 80 workers, KV gate 0.85.
# E128 on the 45 multi-pair eval tasks (E046 run 3): pass@2 83.1% vs 77.5% per pair at -29% tokens per task.
# ARC_CONTEXT_LIMIT = the server's max-model-len: turns never request past it (a 3-input task overflowed in run 3).
import json, os
# B32-M + Manh's server: his server (cell 2m) has B32-M's max-model-len 163840 and max-num-seqs 80 (= the 80 workers)
# on port 18084, so the context limit is its max-model-len. The solver never starts or restarts it; his supervisor
# (cell 2m) restarts it after an exit, ARC_MAX_SERVER_RESTARTS times at most (B32's 3), and the solver follows (S26).
# ARC_MAX_NUM_SEQS / ARC_GPU_UTIL / ARC_SERVER_ENV_JSON / ARC_SERVER_EXTRA_ARGS are gone: nothing reads them.
# The model is still the attached one (cell 1: the dir holding e043_manifest.json; ARC_MODEL_SRC=<dir> to override).
for _k, _v in (("ARC_WORKERS", "64"),
               ("ARC_MULTI_TEST", "1"),
               ("ARC_MULTI_BUDGET", "0.5"),
               ("ARC_CONTEXT_LIMIT", "163840"),
               ("ARC_KV_ADMIT_MAX", "0.85"),
               ("ARC_KV_ADMIT_MIN_WORKERS", "16"),
               ("ARC_AUTO_REPAIR", "1"),
               ("ARC_ADMIT_MAX_WAITING", "1"),
               ("ARC_PORT", "18084"),
               ("MANH_VISION", "0"),  # B32-M fix: text prompts, his --language-model-only path
               ("ARC_MAX_SERVER_RESTARTS", "3")):
    os.environ.setdefault(_k, _v)
print('PYTORCH_CUDA_ALLOC_CONF from the image:', repr(os.environ.get('PYTORCH_CUDA_ALLOC_CONF')), flush=True)
_ac = [x for x in os.environ.get('PYTORCH_CUDA_ALLOC_CONF', '').split(',') if x and 'expandable_segments' not in x]
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ','.join(_ac)
# The commit run ("Save & Run All") is a smoke test: server start + 3 tasks (2 multi-test). The competition rerun
# ignores ARC_COMMIT_MODE (cell 0 always runs "submit") and never sees the task list.
if not os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    os.environ.setdefault("ARC_COMMIT_MODE", "smoke")
    os.environ.setdefault("ARC_METRICS_EVERY_S", "30")  # B32-M: server metrics every 30 s
    os.environ.setdefault("ARC_TASK_IDS", "00576224,25ff71a9")  # E066 light smoke: the 2 smallest tasks
    os.environ.setdefault("E066_LIGHT_SMOKE", "1")  # cells 3w and 4b: the light smoke's checks
    print("E066 LIGHT SMOKE: tasks", os.environ["ARC_TASK_IDS"], flush=True)
    # E066 light smoke: the solver ends once the 2 tasks are done (model load, contract, generation, final write); the deadlines below only cap it at ~75 min of the first cell
    os.environ.setdefault("ARC_STOP_NEW_S", "4200")
    os.environ.setdefault("ARC_FORCE_AT_S", "4320")
    os.environ.setdefault("ARC_HARD_STOP_S", "4380")
    os.environ.setdefault("ARC_KILL_AT_S", "4470")
    import shutil, subprocess, threading, time as _st
    os.environ.setdefault("ARC_SMOKE_ENV_LOG", "/kaggle/working/smoke_env.jsonl")

    def _smoke_env_sampler(path=os.environ["ARC_SMOKE_ENV_LOG"]):
        def _rd(p):
            try:
                return open(p).read().strip()
            except Exception:
                return None
        while True:
            try:
                rec = {"t": round(_st.time(), 1)}
                for k in ("memory.current", "memory.peak", "memory.max", "memory.swap.current"):
                    rec[k] = _rd(f"/sys/fs/cgroup/{k}")
                rec["stat"] = {l.split()[0]: int(l.split()[1]) for l in (_rd("/sys/fs/cgroup/memory.stat") or "").splitlines()
                               if l.split()[:1] and l.split()[0] in ("anon", "file", "shmem", "unevictable", "file_mapped")}
                rec["meminfo_kb"] = {l.split(":")[0]: int(l.split()[1]) for l in (_rd("/proc/meminfo") or "").splitlines()
                                     if l.split(":")[0] in ("MemTotal", "MemAvailable", "Shmem", "Mlocked", "Cached")}
                try:
                    rec["gpu"] = subprocess.run(["nvidia-smi", "--query-gpu=index,power.draw,power.limit,clocks.sm,"
                                                 "memory.used,memory.total,temperature.gpu", "--format=csv,noheader,nounits"],
                                                capture_output=True, text=True, timeout=20).stdout.strip().splitlines()
                except Exception:
                    rec["gpu"] = []
                _sb_n, _sb_kb = 0, 0
                for _p in os.listdir("/proc"):
                    if _p.isdigit() and _rd(f"/proc/{_p}/oom_score_adj") == "1000":  # tool sandbox sessions
                        _sb_n += 1
                        _sb_kb += int(next((l.split()[1] for l in (_rd(f"/proc/{_p}/status") or "").splitlines()
                                            if l.startswith("VmRSS:")), 0))
                rec["sandbox"] = {"n": _sb_n, "rss_gib": round(_sb_kb / 2**20, 2)}
                rec["disk_free_gib"] = {d: round(shutil.disk_usage(d).free / 2**30, 1)
                                        for d in ("/kaggle/working", "/kaggle/temp", "/kaggle/tmp", "/tmp", "/dev/shm")
                                        if os.path.exists(d)}
                with open(path, "a") as f:
                    f.write(json.dumps(rec) + "\n")
            except Exception:
                pass
            _st.sleep(30)
    threading.Thread(target=_smoke_env_sampler, daemon=True).start()
print("E046 settings:", {k: os.environ.get(k) for k in ("ARC_WORKERS", "ARC_MULTI_TEST", "ARC_MULTI_BUDGET",
                                                     "ARC_CONTEXT_LIMIT", "ARC_KV_ADMIT_MAX", "ARC_MAX_NUM_SEQS", "ARC_GPU_UTIL",
                                                     "ARC_REASONING_EFFORT", "ARC_AUTO_REPAIR", "ARC_BEST_GUESS_AT",
                                                     "ARC_BEST_GUESS_TURNS", "ARC_POST_PASS_NOTE", "ARC_ADMIT_MAX_WAITING",
                                                     "ARC_STOP_NEW_S", "ARC_FORCE_AT_S", "ARC_HARD_STOP_S", "ARC_KILL_AT_S",
                                                     "ARC_SERVER_EXTRA_ARGS", "PYTORCH_CUDA_ALLOC_CONF")}, flush=True)

In [ ]:
# Cell 0: clock, paths, fallback submission (before anything that can fail).
import glob, json, os, subprocess, sys, time
T0 = float(os.environ.get("ARC_T0") or time.time()); os.environ["ARC_T0"] = str(T0)
INPUT = os.environ.get("ARC_INPUT_ROOT", "/kaggle/input")
WORK = os.environ.get("ARC_OUT", "/kaggle/working"); os.environ["ARC_OUT"] = WORK
SCRATCH = os.environ.get("ARC_SCRATCH", "/tmp/dsv41"); os.makedirs(SCRATCH, exist_ok=True); os.makedirs(WORK, exist_ok=True)
RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
# Commit runs ("Save & Run All") only check the pipeline: "fallback" writes the fallback and stops; "smoke" runs
# ARC_TASK_LIMIT tasks with short deadlines. The competition rerun always runs "submit".
COMMIT_MODE = os.environ.get("ARC_COMMIT_MODE", "fallback")
MODE = "submit" if RERUN else os.environ.get("ARC_MODE", COMMIT_MODE)

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None

# Every cache and HOME under SCRATCH, overriding the image's settings: with a model attached, /root/.cache is
# read-only on Kaggle (smoke v2 and bench v2: uv "Read-only file system" in /root/.cache/uv; setdefault kept the
# image's UV_CACHE_DIR). HOME is changed only for the subprocesses (install, solver, server), not this kernel.
_C = f"{SCRATCH}/cache"
SUB_HOME = f"{SCRATCH}/home"
os.makedirs(SUB_HOME, exist_ok=True)
for _k, _v in (("UV_CACHE_DIR", f"{_C}/uv"), ("XDG_CACHE_HOME", _C), ("VLLM_CACHE_ROOT", f"{_C}/vllm"),
               ("TRITON_CACHE_DIR", f"{_C}/triton"), ("TORCHINDUCTOR_CACHE_DIR", f"{_C}/inductor"),
               ("FLASHINFER_WORKSPACE_BASE", _C), ("TORCH_EXTENSIONS_DIR", f"{_C}/torch_ext"),
               ("HF_HOME", f"{_C}/hf"), ("CUDA_CACHE_PATH", f"{_C}/nv")):
    os.environ[_k] = _v
    os.makedirs(_v, exist_ok=True)
os.environ.pop("UV_NO_CACHE", None)
CHALLENGES = os.environ.get("ARC_CHALLENGES") or find_one(f"{INPUT}/**/arc-agi_test_challenges.json")
os.environ["ARC_CHALLENGES"] = CHALLENGES
_tasks = json.load(open(CHALLENGES))
_fb = {t: [{"attempt_1": x["input"], "attempt_2": x["input"]} for x in v["test"]] for t, v in _tasks.items()}
with open(f"{WORK}/submission.json.tmp", "w") as f:
    json.dump(_fb, f)
os.replace(f"{WORK}/submission.json.tmp", f"{WORK}/submission.json")
print(f"mode {MODE}; fallback submission for {len(_tasks)} tasks written at +{time.time() - T0:.0f}s", flush=True)
# The kit dataset is one tarball (Kaggle datasets do not keep nested folders reliably): unpack and check sha256.
KIT = os.environ.get("ARC_KIT")
if not KIT:
    try:
        import hashlib, tarfile
        _tgz = find_one(f"{INPUT}/**/dsv41_kit.tgz")
        KIT = f"{SCRATCH}/kit"
        with tarfile.open(_tgz) as tf:
            tf.extractall(KIT)
        _man = json.load(open(f"{KIT}/MANIFEST.json"))
        _bad = [f for f, h in _man.items() if hashlib.sha256(open(f"{KIT}/{f}", "rb").read()).hexdigest() != h]
        assert not _bad, f"kit files differ from MANIFEST: {_bad[:5]}"
        print(f"kit unpacked from {_tgz}: {len(_man)} files, sha256 ok", flush=True)
    except Exception as exc:  # without the kit nothing else can run: keep the fallback submission
        print("kit unavailable, fallback only:", type(exc).__name__, exc, flush=True)
        MODE = "fallback"
MODEL_SRC = os.environ.get("ARC_MODEL_SRC") or (os.path.dirname(find_one(f"{INPUT}/**/e043_manifest.json") or "") or None)
print("kit", KIT, "| model", MODEL_SRC, flush=True)

In [ ]:
# Cell 0a (B32-M GPU gate): fail fast when the session has no usable GPU or the wrong Python, before Manh's install.
# Smoke V1 (2026-10-28): every nvidia-smi sample said "couldn't find libnvidia-ml.so" and torch saw 0 GPUs. Records
# /dev/nvidia*, nvidia-smi -L, ldconfig's NVIDIA libraries, NVIDIA_* / LD_LIBRARY_PATH, every libnvidia-ml.so* /
# libcuda.so* under /usr and /opt, and the Python version (b32m_gpu_gate.json). If the device nodes exist but NVML is
# not found while a libnvidia-ml.so.1 exists, its folder goes on LD_LIBRARY_PATH (os.environ: subprocesses inherit
# it). It retries for up to ARC_GPU_GATE_S (90 s; NVML has appeared ~30 s after start before). No 4 GPUs, or a Python
# other than 3.13 (his wheels are cp313): commit runs RAISE at once (no quota burnt); the competition rerun keeps
# B32's fallback submission and turns every later cell of ours off (MODE = "fallback").
import json, os, subprocess, sys, time


def _gate_sh(cmd, timeout=60):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, errors="replace", timeout=timeout)
        return r.returncode, ((r.stdout or "") + (r.stderr or "")).strip()
    except Exception as exc:
        return -1, f"{type(exc).__name__}: {exc}"


def _gate_gpus():
    rc, out = _gate_sh("nvidia-smi -L", 30)
    return sum(1 for l in out.splitlines() if l.startswith("GPU ")), rc, out[-600:]


GPU_GATE = {"t0_s": round(time.time() - T0), "python": sys.version.split()[0],
            "dev_nvidia": _gate_sh("ls -l /dev/nvidia* 2>&1")[1][-1500:],
            "ldconfig_nvidia": _gate_sh("ldconfig -p | grep -i nvidia")[1][-2500:],
            "env": {k: v for k, v in os.environ.items() if k.startswith("NVIDIA_") or k == "LD_LIBRARY_PATH"},
            "libs": _gate_sh("find /usr /opt \\( -name 'libnvidia-ml.so*' -o -name 'libcuda.so*' \\) 2>/dev/null "
                             "| sort", 120)[1].splitlines()[:60], "tries": []}
_need = int(os.environ.get("ARC_GPU_GATE_N", "4"))
_t = time.time()
while True:
    _n, _rc, _out = _gate_gpus()
    GPU_GATE["tries"].append({"t_s": round(time.time() - T0), "gpus": _n, "rc": _rc, "out": _out[-300:]})
    if _n >= _need:
        break
    if ("libnvidia-ml" in _out or "NVML" in _out) and GPU_GATE["dev_nvidia"].count("/dev/nvidia") \
            and not GPU_GATE.get("ld_added"):
        _ml = [p for p in GPU_GATE["libs"] if os.path.basename(p) == "libnvidia-ml.so.1"]
        if _ml:  # NVML exists but is not on the loader path: put its folder first, for this kernel and subprocesses
            _d = os.path.dirname(_ml[0])
            os.environ["LD_LIBRARY_PATH"] = _d + (":" + os.environ["LD_LIBRARY_PATH"]
                                                  if os.environ.get("LD_LIBRARY_PATH") else "")
            GPU_GATE["ld_added"] = _d
            continue
    if time.time() - _t > float(os.environ.get("ARC_GPU_GATE_S", "90")):
        break
    time.sleep(10)
GPU_GATE["gpus"] = _n
_why = []
if _n < _need:
    _why.append(f"{_n} GPUs visible to nvidia-smi after {time.time() - _t:.0f}s (need {_need}): {_out[-200:]!r}")
_py = os.environ.get("ARC_GPU_GATE_PY", "3.13")  # his wheels are cp313
if ".".join(map(str, sys.version_info[:2])) != _py:
    _why.append(f"Python {GPU_GATE['python']}, but Manh's wheels need {_py} (cp313)")
GPU_GATE["fail"] = _why
with open(f"{WORK}/b32m_gpu_gate.json", "w") as _f:
    json.dump(GPU_GATE, _f, indent=1)
if _why:
    print("=" * 100, f"B32-M GPU GATE FAILED: {_why}", json.dumps(GPU_GATE)[:3000], "=" * 100, sep="\n", flush=True)
    if not RERUN:
        raise RuntimeError(f"B32-M GPU GATE FAILED (commit run): {_why}")
    MODE = "fallback"  # rerun: B32's fallback submission (written by cell 0) stands; our later cells do nothing
else:
    print(f"B32-M GPU GATE OK: {_n} GPUs, Python {GPU_GATE['python']}, LD_LIBRARY_PATH added:",
          GPU_GATE.get("ld_added"), flush=True)

In [ ]:
# Cell 0b (E046, frozen inputs): the pinned solver + harness ride in the notebook, so the inputs stay exactly
# Version 5's (no new dataset version to mount). Overwrites the kit's two files after cell 0 unpacked and checked it,
# then checks hashes, required features and settings.
# Commit run ("Save & Run All"): any failure RAISES, so the commit errors and the log shows "E046 CHECK FAILED".
# Competition rerun: the same failure restores the kit's original v3 files and runs per pair at 36 workers (a scored
# submission beats an error); the log line is printed either way. The rerun path never raises (S10): a file whose
# backup is missing or cannot be copied back keeps what is in the kit (the pinned file, or the kit's own if the
# overlay never ran), and the notebook still runs.
import base64, hashlib, io, shutil, tarfile
_MT = "..."
_MT_SHA = {"solver_dsv41.py": "324ea6ee13f96f6c2996371b4ca0843d75026826fc6f8bdf6cf7a5f41d0da740", "repo/evals/run_deepseek_tools.py": "882c817f2c93dd8e9ebf65c241a694549c5b7679cbacca5d16906a28dd62eecc", "repo/evals/train_check.py": "8fe361752010bba7599b24d83c4491ee462a2cffb1d9b1d018c67b90901fc982", "repo/evals/auto_repair.py": "2292369c4ad26fea4e2b3dc5e82dad31d1180192510484a4a1021a7e89b8a33d", "repo/evals/auto_repair_names.json": "571f47eb1bd077358d2153a943d0e6710c4ec18cf5828174d86c8be1fac03e92"}
_MT_EXPECT = {"ARC_WORKERS": "64", "ARC_MULTI_TEST": "1", "ARC_MULTI_BUDGET": "0.5", "ARC_CONTEXT_LIMIT": "163840", "ARC_KV_ADMIT_MAX": "0.85", "ARC_AUTO_REPAIR": "1", "ARC_ADMIT_MAX_WAITING": "1", "MANH_VISION": "0"}
_MT_NEW = ["repo/evals/auto_repair.py", "repo/evals/auto_repair_names.json"]
_MT_NEED = {"repo/evals/run_deepseek_tools.py": ["--auto-repair"], "solver_dsv41.py": ["ARC_ADMIT_MAX_WAITING", "ARC_EXTERNAL_SERVER", "ARC_EXTERNAL_SERVER_STATUS", "EXTERNAL = E(\"ARC_EXTERNAL_SERVER\"", "EXTERNAL_STATUS = E(\"ARC_EXTERNAL_SERVER_STATUS\""]}
if MODE != "fallback":
    _bak, _why = {}, ""
    try:
        assert KIT, "kit not unpacked"
        for _f in _MT_SHA:
            if _f in _MT_NEW:
                continue  # new to the kit (no v3 original); the v3 code never imports it
            shutil.copy2(f"{KIT}/{_f}", f"{KIT}/{_f}.v3orig")
            _bak[_f] = f"{KIT}/{_f}.v3orig"  # S10: only once the backup exists
        with tarfile.open(fileobj=io.BytesIO(base64.b64decode(_MT))) as _tf:
            _tf.extractall(KIT)
        for _f, _h in _MT_SHA.items():
            assert hashlib.sha256(open(f"{KIT}/{_f}", "rb").read()).hexdigest() == _h, f"overlay mismatch: {_f}"
        _sv = open(f"{KIT}/solver_dsv41.py").read()
        _hv = open(f"{KIT}/repo/evals/run_deepseek_tools.py").read()
        for _need in ("MULTI_TEST", "MULTI_BUDGET", "CONTEXT_LIMIT", "KV_ADMIT_MAX", "split_multi", "restart_allowed",
                      "shutdown_socket"):
            assert _need in _sv, f"solver lacks {_need}"
        for _need in ("--multi-test", "--multi-budget", "--context-limit", "class ContextFull", "import builtins as _b",
                      "def answer_items", "def context_budget"):
            assert _need in _hv, f"harness lacks {_need}"
        for _f, _needs in _MT_NEED.items():  # features the chosen opt-in settings rely on
            _txt = open(f"{KIT}/{_f}").read()
            for _need in _needs:
                assert _need in _txt, f"{_f} lacks {_need}"
        for _k, _v in _MT_EXPECT.items():
            assert os.environ.get(_k) == _v, f"{_k}={os.environ.get(_k)!r}, expected {_v!r}"
    except Exception as exc:
        _why = f"{type(exc).__name__}: {exc}"
    if not _why:
        print("E046 CHECK OK: multi-test code installed and checked:", {f: h[:8] for f, h in _MT_SHA.items()},
              _MT_EXPECT, flush=True)
    else:
        print("=" * 100, f"E046 CHECK FAILED: {_why}", "=" * 100, sep=chr(10), flush=True)
        if not RERUN:
            raise RuntimeError(f"E046 CHECK FAILED (commit run): {_why}")
        _kept = []
        try:
            for _f in _MT_SHA:  # rerun: back to the kit's v3 files, per pair, 36 workers
                if _f in _MT_NEW:
                    continue
                try:
                    shutil.copy2(_bak[_f], f"{KIT}/{_f}.restore")
                    os.replace(f"{KIT}/{_f}.restore", f"{KIT}/{_f}")
                except Exception as exc:
                    _kept.append(f"{_f} ({type(exc).__name__})")
            # (a kept pinned solver would pass the opt-in harness flags to the v3 harness, whose parser exits)
            for _k in ("ARC_MULTI_TEST", "ARC_MULTI_BUDGET", "ARC_KV_ADMIT_MAX", "ARC_CONTEXT_LIMIT", "ARC_AUTO_REPAIR",
                       "ARC_BEST_GUESS_AT", "ARC_BEST_GUESS_TURNS", "ARC_POST_PASS_NOTE", "ARC_REASONING_EFFORT",
                       "ARC_ADMIT_MAX_WAITING"):
                os.environ.pop(_k, None)
            os.environ["ARC_WORKERS"] = "36"
            os.environ["ARC_SERVER_EXTRA_ARGS"] = "[\"--linear-backend\", \"humming\", \"--compilation-config\", \"{\\\"cudagraph_mode\\\": \\\"FULL\\\", \\\"max_cudagraph_capture_size\\\": 64}\"]"  # no flag that needs the new overlay
        except Exception as exc:
            _kept.append(f"{type(exc).__name__}: {exc}")
        print("E046 FALLBACK: the kit's v3 code, per pair, 36 workers"
              + (f"; not restored, kept as is: {', '.join(_kept)}" if _kept else ""), flush=True)


In [ ]:
# Cell 1: stage the checkpoint to local disk in the background (stage_model.py), while vLLM installs.
# "warm" (read the mount once, no writes): smoke v2 (2026-10-24) with "copy" hit "Read-only file system" on the
# root overlay during the install while 131 GB were being written to /tmp.
STAGE_MODE = os.environ.get("ARC_STAGE_MODE", "warm")
STAGE_DONE = f"{SCRATCH}/stage_done"
if MODE != "fallback" and MODEL_SRC:
  try:
    if os.path.exists(STAGE_DONE):
        os.remove(STAGE_DONE)
    _stage = subprocess.Popen([sys.executable, "-I", f"{KIT}/stage_model.py", MODEL_SRC, f"{SCRATCH}/model", STAGE_DONE,
                               STAGE_MODE, os.environ.get("ARC_STAGE_THREADS", "16")],
                              stdout=open(f"{SCRATCH}/stage.log", "w"), stderr=subprocess.STDOUT,
                              stdin=subprocess.DEVNULL, start_new_session=True)
    print(f"staging ({STAGE_MODE}) started, pid {_stage.pid}", flush=True)
  except Exception as exc:  # the solver then loads from MODEL_SRC (no staging marker)
    STAGE_DONE = ""
    print("staging failed to start:", type(exc).__name__, exc, flush=True)

In [ ]:
# Cell 1b (smoke, commit runs only): input integrity, in a background thread so it costs no wall time. Checks every
# input the competition rerun reads: the competition JSON files parse; the kit (already sha256-checked by cell 0); the
# offline wheel set against its SHA256SUMS when it ships one; and every file of the model (136 files, 131 GB) by size
# and sha256 against the manifest made before upload. The model is hashed once the server is up, so it does not slow
# the load. Results go to /kaggle/working/smoke_integrity.json; the report cell prints them and fails the commit run on
# any mismatch. The competition rerun skips this cell. Uses cell 0's T0, INPUT, WORK, KIT, MODEL_SRC, MODE, RERUN and
# find_one.
import hashlib
import json
import os
import threading
import time
from concurrent.futures import ThreadPoolExecutor

INTEGRITY = json.loads(r"""{"model": {"LICENSE": [1084, "f2c6c602815669d292889e5be8c802f2ed950653b77999b1584e8e6aed25d040"], "README.md": [4232, "80ea4ac33d69c27c3a3a51692ccbeb745ac5c71c8cfa1de6737492f9b2ef54e7"], "config.json": [5193, "f85a7c94bf42b4e25e16ebadb2de50480d41fed054e8b908e0d0fc284534bed4"], "e043_manifest.json": [40555, "0203d5074bad368e3e90195cb8132a7e70fb393f01e70a3075f97af13813e536"], "encoding/README.md": [12164, "a34a5ca98c65e5b5b2bedd693cc2e18e732fe50192723fb6937ae07219c23739"], "encoding/encoding.py": [37962, "13231c592e958fb39e2af6001dae69d90e7a36b265702b9b390236fb387599e2"], "encoding/test_encoding.py": [22449, "288b79c9ea7d018ae03b0dd3e54a8220cc3a2df1ff97f59907e75ed8112765d6"], "encoding/tests/test_input_1.json": [2761, "6a5a86694e47169c8f0c9243a7e24cc5db05447be0d3d3a9891034e913c518de"], "encoding/tests/test_input_2.json": [527, "fd985cda88e8640b8e2404d8e659f7006cad204eb41f73c9e6662b1f218c01aa"], "encoding/tests/test_input_3.json": [2554, "4207613f15c79b95ad92074bdb8b2b81c5ce9cfdf15b7a67f58ded6bb3ec4fa5"], "encoding/tests/test_input_4.json": [712, "4c3890a7bc042ec6396d98d024a39a73cdcffbe1d0253e852da0c35e65296f3e"], "encoding/tests/test_input_5.json": [1143, "88289b0aac8d650aa55c4ffeb392818fff3a0ac83c2293ddfd90546804d5e5c4"], "encoding/tests/test_output_1.txt": [2476, "91f79aa955717b414a82aabfb38bcd83ab5a998bd47db89d2a26338e8d3de550"], "encoding/tests/test_output_2.txt": [294, "9fa162587b73ab9b315be5c88d433f9e0e681b4b36926952f289099ed9bc1dab"], "encoding/tests/test_output_3.txt": [2481, "cf2b801d0738c88abb691bf4a096000c0e0737faab8a54a4ba5243c5513a3711"], "encoding/tests/test_output_4.txt": [574, "94433b68370dcc69fa61efc5011b5d0e28685e4716fee1ee77fbbfc0c9935b54"], "encoding/tests/test_output_5.txt": [408, "4d673091746553d1cb2e40a3fc01dc09a7c53c40627096a3e49c97a269a81f64"], "gptq/layer_00.json": [34363, "8070c088054bf4b1430486d8745e523c24ace9b2461a89fee4c6e84f08d2cf78"], "gptq/layer_00.safetensors": [1238726280, "0478e50eddbc304ee75c109647f2ac7847844689d2d267c4704759cfeba676d9"], "gptq/layer_01.json": [34308, "f924c25dc7fa0801bd0280f3730a9ea3075f30a0e2039c59b598be899d5c4da6"], "gptq/layer_01.safetensors": [1238726280, "3a306a2be5a069efe28c938d2b6e62dc8b2390d1d8f217c2bdb2ae60bd8f5f53"], "gptq/layer_02.json": [34337, "0e4bff8fc8f9b118c623a5eaaf4f1ae2dc7890a41bb45ce6f73632f2dcb8f1f9"], "gptq/layer_02.safetensors": [1238726280, "064c0760d7d803625ce1e7ce7b63de4e42ad7b289b42a8b64f968ff477e94f46"], "gptq/layer_03.json": [34308, "8e9fc5d8a0c95475b1965ebb03a8581d3ea73dc294508fb6a2fe3064028cd3df"], "gptq/layer_03.safetensors": [1238726280, "f1765866a9a98e09199363ddbf7e2741e8d20af2fa87e66f4b3e0a4ea0f6c9e6"], "gptq/layer_04.json": [34284, "7c02de8a310e4dfb6b846487096a256b5585925eecfd0e1f53959e28f92008c6"], "gptq/layer_04.safetensors": [1238726280, "1f83ee0c0ea45ee5fadb53bf2efbdac9cc1c85fe964b47ad59060e25b3cacf3a"], "gptq/layer_05.json": [34245, "4b210e637e46f7a606c8217162b3843e3c917e1880d904b357add4c06777bc17"], "gptq/layer_05.safetensors": [1238726280, "1f586d333951528d4bf1a0a65315578acf2cd6caea78ecf6b5566362bc639c86"], "gptq/layer_06.json": [34295, "84257b1f05474d6c06831e5f7a9f203af207734c4332a8400159d434d9c37bc7"], "gptq/layer_06.safetensors": [1238726280, "52bb959b015f1451c1fa604fc72e143754a4390982650489d3c16ce3e1e00ab3"], "gptq/layer_07.json": [34236, "e6c3ba1e042c12564082e67d25e693fa1dd5cd0d6518a4efae9811afec10ad8e"], "gptq/layer_07.safetensors": [1238726280, "6fe0738fd41e59d6b3ed45dd91711d29a16317fd553d4354c1041fa06e09776f"], "gptq/layer_08.json": [34259, "8ee6d15c002061526cd75bb6e06940a2281750b16d3f7dafb7dc64e09a7e369b"], "gptq/layer_08.safetensors": [1238726280, "fa2f89c9953d62cb50a210abc5ac9b99944b61396f65d1a917c3a765e52e7f1b"], "gptq/layer_09.json": [34274, "653bbd7b04ace39e1243880f0d0a106dbeb3c147b8cf82f121cadb8d8d8fda56"], "gptq/layer_09.safetensors": [1238726280, "1d58f087f9f210e4a9521f6f2dacba298f67e1e52e54f8751d8bac72b539b736"], "gptq/layer_10.json": [34251, "1f275848abcb0997eab0a157fca49218f67e07a51538965acc4d1c13dd4cbf5a"], "gptq/layer_10.safetensors": [1238726280, "2c1f30fb6ac72a5da1b728a720ec07b9ef405d05b210a6c2aee66aa1525fac61"], "gptq/layer_11.json": [34219, "67c3369fa913a2551eb285220cd58e80d6d967440b5d23d48eed3d6d23c29499"], "gptq/layer_11.safetensors": [1238726280, "494098074b791e1f98aae982211f510f27a83214ddb8dbab5a866adef50c6be6"], "gptq/layer_12.json": [34203, "f1e22cd9ddb9a2673298c5895f8b119429ef3228e4dc408ec122813b4bdd989e"], "gptq/layer_12.safetensors": [1238726280, "1475a262128907a0ea07537c73d3ed084b1c05241cbb0299c52b09d5ee1c5b1b"], "gptq/layer_13.json": [34227, "8b5fb5922b631dd98d3c163eed5fdb8f96214b026c56a2a927abd370d3b27eda"], "gptq/layer_13.safetensors": [1238726280, "0671ef29a3be767714bd44f674dd728a1d0856823ae9612e1a72c6c6fc1e2fce"], "gptq/layer_14.json": [34231, "90686ff97447f0b7ad3cb610e7893166d63366580577abfe65ae7923b3816999"], "gptq/layer_14.safetensors": [1238726280, "3f2ba1bd776811e8cfcf4eebcff2561fec7b9fe3c3ed9297e3f348f0a9af6041"], "gptq/layer_15.json": [34237, "f4574048fd6223c4f5b3f1350cc664e2e00d411541e013d64ea6ff862cdceb7e"], "gptq/layer_15.safetensors": [1238726280, "679b0c96c7d2ffee56bdc69c49eb27c85a2900141de1cf9ab2634c9932712acf"], "gptq/layer_16.json": [34199, "9cca7ea1727a2010def8d4ba070c9dc56b538c83c7c84dc7628fe7c2bbc8d771"], "gptq/layer_16.safetensors": [1238726280, "e292f2b6608fdc21af668abf3ce4504bd1d106ae727b658c75e0b2878c19fe4c"], "gptq/layer_17.json": [34190, "c09dd881c7b44791218b91ccddcf633fa36af6b91142c0b102acf5ee50d1ef6c"], "gptq/layer_17.safetensors": [1238726280, "1f6bd28a1de032aa1823865c50484066d2c27af7d757f8953e909f5e35ed761a"], "gptq/layer_18.json": [34215, "d5f5c809f786ef996a353debab33ba255e1c1543523704c6023978df9583a43e"], "gptq/layer_18.safetensors": [1238726280, "5222a1ed0d40563cc7ea6d8eb403d5e99a404f5c1476fda99b7469fff809f819"], "gptq/layer_19.json": [34227, "f36b03ed1be451a91113c5387f4443588ee2990d7b07ebe530b18fcbcd4e08e0"], "gptq/layer_19.safetensors": [1238726280, "144b7c9c51db9def462c05fdf469bd77a6bba6474ae65fce03993e4abaa18398"], "gptq/layer_20.json": [34216, "4f657799163609e1491cac8a424e2a330da61980f01d454904b8c5a41df6bd58"], "gptq/layer_20.safetensors": [1238726280, "174548e63b57648de373e8d78bec0c21558885d365bfbf38ed06a3b2318ff8fd"], "gptq/layer_21.json": [34203, "a98843417ed0d2065f08d406f279154da9981dfde9a80e8b02255c08427bf49f"], "gptq/layer_21.safetensors": [1238726280, "c8f057644083a2f01b0b121aaf7dcb6754907254b6dca32ac8b305a18ff46a10"], "gptq/layer_22.json": [34197, "d674cae5d82d88115561a36d50c9c6ce8ad3f2de609131ca038a2dd5f9892f34"], "gptq/layer_22.safetensors": [1238726280, "286ca8ee5a1f7cc92c90bf3e31f94d068eea09bb3da56f0cf5e08a642a9ee7dd"], "gptq/layer_23.json": [34234, "038b71cb3890fbaef35adf5314ecb5006681a2299342db94cb3b2258078edb28"], "gptq/layer_23.safetensors": [1238726280, "2c0d8f231a67e941d903e7f24d4fdef62747d5a53faff0daa02e6eb28c8cf659"], "gptq/layer_24.json": [34200, "30d38fbf20ea46de55d8da98358d9965ffe1ba2aa05df45e9a2d7eb24712dd7c"], "gptq/layer_24.safetensors": [1238726280, "d50027403a14ecc7ce46befed27b3d8d17c8f2aab20d078221e10a550a1d90cd"], "gptq/layer_25.json": [34177, "9fe1951e2ade0de1c881a893d079b816463ec99a64c9e40de04281f9b6d4b32a"], "gptq/layer_25.safetensors": [1238726280, "31d744cf34b945aeeb6159d612763a78694e031ffa7c88d28253ffa7d448f770"], "gptq/layer_26.json": [34167, "85b6007be2594c3633c86ac296d60357bf6d4546d29a861b0adc8df3b398702c"], "gptq/layer_26.safetensors": [1238726280, "950d25f1662eb5f94bbbb40aaa48efe64b614efce75552d3ef94956aff9e74bc"], "gptq/layer_27.json": [34196, "9af152d423070d986baeb82de18eb4081a4c63c06f78e60b7d3ee21193cefc27"], "gptq/layer_27.safetensors": [1238726280, "d197aa4c96ae2d8217d2e3876d6a7be01b07fd57c96458223848bfad17d3fc94"], "gptq/layer_28.json": [34177, "d31aea2be9ad421e0da3573caa8700bad75c29daa835fc5a030dbec1a836369a"], "gptq/layer_28.safetensors": [1238726280, "a05c5e62022ae4c5ed87c4d80c5b99f6b45e3c2cc57ade83346914b4ab58c590"], "gptq/layer_29.json": [34183, "7f6aae2fc4fff26b532eb4d304f21ff19d671f7b270a0aeb9bec5b898c8d1945"], "gptq/layer_29.safetensors": [1238726280, "8dfc815323471a19eea6be0fb4fa400c3609f2e63e151d41794682947536fb65"], "gptq/layer_30.json": [34165, "5f75bdeb910c779b8d0392b829662ad0b29e5e75268bcbf093ede29c8262139f"], "gptq/layer_30.safetensors": [1238726280, "84d0b70e0244235989679731649d65f803708c6e6447c09e0755d0dd1d33ecd1"], "gptq/layer_31.json": [34187, "a09043e40fe345082cfe8843cfe398274273f580f1dab2d25b01cc0328719d01"], "gptq/layer_31.safetensors": [1238726280, "05dd7b0e9e401c16b8e8f6432c3ea45cf01b9653aad8f6c682a210b0b6681dea"], "gptq/layer_32.json": [34186, "3f9c578268c9ad5682e4b10189a011192b762bdcab3c576d5e17322e5fe9074e"], "gptq/layer_32.safetensors": [1238726280, "f58e77c143b7bb549f449881c6e9f464859feaf7a4976d018b7426fcda1e063d"], "gptq/layer_33.json": [34145, "b4cbb0ebb96aa92cdd38f05aa08125294fa00bd6877a7b48b038d58979cf6b82"], "gptq/layer_33.safetensors": [1238726280, "4c2cf38a6d3b9fc61204112cea1db511ba4e4cd9869a8e00b9a97a72a1eec6ce"], "gptq/layer_34.json": [34159, "f55882c644eeb48dee59df10c99636f228097b8f31954ce5ce4f5f5a1e3d05ac"], "gptq/layer_34.safetensors": [1238726280, "af5691a717caf00971e4c29589bc9af992dde36f449be431484ae35a83102c8e"], "gptq/layer_35.json": [34229, "d9eba6ecb3c3338c2efdcc2df6bf4ebd7a4bad864ae4c294eec2e2a7ef7ddb4f"], "gptq/layer_35.safetensors": [1238726280, "1076ae8a2105848452b91e8b25d9490faf8be7ebe36ccef41d5c586191d20820"], "gptq/layer_36.json": [34229, "e9246f13555a3706a55cab4e181d3cf80c6c40050b3dc9b0873ec22504ce0825"], "gptq/layer_36.safetensors": [1238726280, "3f75926226d0ea0767c3f13984df112f2b3ba48c32c6d0bf87c3975ab7607521"], "gptq/layer_37.json": [34242, "bef7243980a9b24df76017d93f77b5c5c82ee17956883007045a8d0b3cff6097"], "gptq/layer_37.safetensors": [1238726280, "eec73875ba790e52cb889d35d38cfe2ab7d00e4758f04f39229ef62e4e6905be"], "gptq/layer_38.json": [34233, "bd2f44aedcc211f67dac637fabb891749166336eca6fba461fe0c3a6673eeab2"], "gptq/layer_38.safetensors": [1238726280, "f2999881df39f3e34e6b961c8dcd69602e6c2fa33e33867637207e6416ad7ae6"], "gptq/layer_39.json": [34321, "94a619f52806d0c9491959419007071278fcc7184806c75162863b09eef84a3d"], "gptq/layer_39.safetensors": [1238726280, "e0b9891b75d1238c669709d68424ac10cdd16b1e9f9fa89fc20cb847b976c80b"], "inference/README.md": [2029, "2834402823199ee24e9a42bdf36a0fc6daf94448f444cb062c042a057a798f1c"], "inference/config.json": [1982, "2e84f45cf1dac8c7fcbb200e96667d4b913275690668ed496f24c7747207a809"], "inference/convert.py": [9458, "035028340479145594a81d6084a8424e57363adf83c0d5983914783d95614d76"], "inference/engram.py": [8138, "11f35ecbead8150c35aa002b3d180ef290b05a25afe883a11884f94d476d3897"], "inference/examples/example.txt": [332, "8456a2f1d5a15b7aa1f697567e34f79af0522fb694df7ac789dae9853fff165d"], "inference/examples/example_harmony.json": [2156, "44ae84a3cbd360b1b8d303ed4006756afb77e5d014c4abb7c61ff9b9a094f6ea"], "inference/examples/images/carrots.jpeg": [212495, "5df896a4a07e127281c60fc957f8b3d73f4735b3258a0bf762b4383557f8fa9a"], "inference/examples/images/corn.jpeg": [56122, "251fe7ad58f44db139a6af5ed489e027eca93ae60878b83f3e709c7d6f0112a1"], "inference/generate.py": [8722, "8668d67f7d108e32b90d50cb0d8606889ceb2219bfe95741d84e22f70768e9f0"], "inference/image_processor.py": [7699, "482759e3bcc4e9bb5ee582b244cc563f5d0e163d8b48dda91ebb7106e62f9272"], "inference/kernel.py": [23790, "1236c3507019ed176f5dba5e04bcea58867cf654818c6cf138ed4845398c2455"], "inference/model.py": [61669, "e5decd76fe70fd56ef2f10124d245c6f64fb6a0003cc9068a1d3182a168b8d2d"], "inference/requirements.txt": [98, "be82bcd5775ccf8361ddd22cac53ef7b3b83434697fc3c61c75870450fb9437e"], "inference/run.sh": [1759, "91eddd7895f9fcd9bace53c21128c506e741e770f3c1a02464eff51f60401dfb"], "inference/vision.py": [4457, "5d49edc196a4ef22384abe76d35a40098cbe1e74b586c8f66a2edff4f076b26c"], "model-00001-of-00002.safetensors": [4990846656, "183eccf1215a92250c8102d1a757beef17a494a3e2c396729b7a84b53e857b9d"], "model-00002-of-00002.safetensors": [4751091632, "78cfb0103dd6918f9aca441c7876eede73faca3928b54f312c2d55efcb867ee1"], "model-engram-00001-of-00016.safetensors": [1088919216, "55c00ea48a06a67028a7c5eeb3897ae8e7f4119b0de0d1c2c8088b2674ded583"], "model-engram-00002-of-00016.safetensors": [4998951576, "fd43f6cc023ff267c37085ccecb074305d235fb3a23747d3bc99d22db6a81ef3"], "model-engram-00003-of-00016.safetensors": [4998951584, "3fa7839ef761a09e7937b78aa917c706570bd2309fb0a410104880a46cb3f05c"], "model-engram-00004-of-00016.safetensors": [4998951584, "1c897280d993b03794cd6d14712e2f995c5a683935e686d1f0d6c43920e1e13a"], "model-engram-00005-of-00016.safetensors": [4998951584, "d214812b29e538caaa0831ece23c636690699587763a9458ad44ce04bce513ef"], "model-engram-00006-of-00016.safetensors": [4998951584, "261a5fef67de77a7877395737f783b4adaadb7c9c0098019bd7cfda95afa4804"], "model-engram-00007-of-00016.safetensors": [4998951584, "f781ae4a959389211dbf92f4ad248c52c4de675d59d71f32a87aa59bb200809b"], "model-engram-00008-of-00016.safetensors": [4851701920, "c1d6d4993c9316a331464f9d07f97cbb45d6b68ff58dcb370e574a8a08bfecaa"], "model-engram-00009-of-00016.safetensors": [2529514248, "5de637f1aaecb973114866a6af06a67ffcafed5c7ab7f21a1746b93c86e5a66a"], "model-engram-00010-of-00016.safetensors": [4998951576, "5a5d3e532bf817a2753685f133ab1144b07c0771f0a5389d44033e62c656e13e"], "model-engram-00011-of-00016.safetensors": [4998951584, "3560c30b277c7cfcf826ba3d9ff38212fcdc7703f087a858f0ccdd914e3e427f"], "model-engram-00012-of-00016.safetensors": [4998951584, "041646d1e4c7ae4e4c8473504ed0777dda00278b3210532b449693964add675b"], "model-engram-00013-of-00016.safetensors": [4998951584, "5e9f091824203c027f2cdeba6d294de700dd2587b452a9cbfcdc97f81125f6f9"], "model-engram-00014-of-00016.safetensors": [4998951584, "32c53926732fa89c634147a5048c44221ed6851f7915f6db85045a1cebad3360"], "model-engram-00015-of-00016.safetensors": [4998951584, "e6c9a6bed470838bc472519603f53646e2b727c64b937a12b3ca2f2e67850851"], "model-engram-00016-of-00016.safetensors": [3334016448, "69b910b6c8e4c1c98d4a117b0cf844218f10e6b07b72346386e0e7752dd548b8"], "model.safetensors.index.json": [90861, "62c48a61621903219dfe74520feb887ec5bcef330b2027ae801d68d7a1ae7931"], "serving/apply_compact_engram.py": [14059, "fd3ef2f9d51eb2e44b572db783b0b384952b0e3319fed74c9e147cac5aa92f17"], "serving/apply_e043_humming.py": [10843, "991074c8f2a47834d98b369eee757b0e86a777802c811fc47e05f7bb522f787b"], "serving/e043_humming.py": [4780, "74dcbb141ff0ad29b56123f1ae24eef8e0249dea48a36dbe76d983a7fa2be8d8"], "tokenizer.json": [6367257, "c90dfa01249db1be4245780a052ede752e1361c612ac6d08e2bdada7d599476b"], "tokenizer_config.json": [801, "6ac8c8dc065ed118161d02dd532749ae3f52c243deac27872134fae2f50d8547"]}}""")  # filled in by build_frozen_mt.py: {"model": {path: [size, sha256]}}
INTEGRITY_OUT = f"{WORK}/smoke_integrity.json"


def integrity_save(state):
    with open(INTEGRITY_OUT + ".tmp", "w") as f:
        json.dump(state, f, indent=1)
    os.replace(INTEGRITY_OUT + ".tmp", INTEGRITY_OUT)


def integrity_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(16 << 20)
            if not block:
                return h.hexdigest()
            h.update(block)


def integrity_worker(state, model_dir, threads=8, wait_s=1800):
    try:
        state["competition"] = {}
        comp_files = sorted({p for p in (find_one(f"{INPUT}/**/arc-agi_test_challenges.json"),) if p})
        comp_dir = os.path.dirname(comp_files[0]) if comp_files else None
        for name in sorted(os.listdir(comp_dir)) if comp_dir else []:
            if name.endswith(".json"):
                with open(os.path.join(comp_dir, name)) as f:
                    state["competition"][name] = len(json.load(f))
        state["kit"] = "sha256-checked by cell 0 (MANIFEST.json)" if KIT else "MISSING"
        wheel = find_one(f"{INPUT}/**/vllm-0.28.1rc1*.whl")
        wdir = os.path.dirname(wheel) if wheel else None
        sums = os.path.join(wdir, "SHA256SUMS") if wdir else None
        if sums and os.path.exists(sums):
            bad = []
            lines = [l.split() for l in open(sums) if l.strip()]
            for digest, name in ((l[0], l[-1].lstrip("*")) for l in lines):
                p = os.path.join(wdir, name)
                if not os.path.exists(p) or integrity_sha256(p) != digest:
                    bad.append(name)
            state["wheels"] = {"checked": len(lines), "bad": bad}
        else:
            state["wheels"] = {"files": len(os.listdir(wdir)) if wdir else 0, "note": "no SHA256SUMS shipped; uv installs "
                               "from the pinned requirements.lock and the import check in cell 2 runs"}
        integrity_save(state)
        # E066: names and sizes of every model file at once (seconds): the sha256 pass starts at server up and
        # rarely finishes inside a light smoke, so cell 4b fails the commit run on a missing or short file
        stat = {"files": len(INTEGRITY["model"]), "missing": [], "size_mismatch": []}
        for rel, (size, _digest) in INTEGRITY["model"].items():
            p = os.path.join(model_dir, rel)
            if not os.path.exists(p):
                stat["missing"].append(rel)
            elif os.path.getsize(p) != size:
                stat["size_mismatch"].append(rel)
        state["model_stat"] = stat
        integrity_save(state)
        log = f"{WORK}/dsv41_logs/solver.log"
        while time.time() - T0 < wait_s:  # hash after the server is up, so the load does not compete for the mount
            if os.path.exists(log) and "server up" in open(log, errors="replace").read():
                break
            time.sleep(15)
        files = INTEGRITY["model"]
        res = {"files": len(files), "checked": 0, "missing": [], "size_mismatch": [], "sha_mismatch": [], "bytes": 0,
               "started_min": round((time.time() - T0) / 60, 1)}
        state["model"] = res
        t0 = time.time()

        def check(rel):
            size, digest = files[rel]
            p = os.path.join(model_dir, rel)
            if not os.path.exists(p):
                return rel, "missing", 0
            if os.path.getsize(p) != size:
                return rel, "size_mismatch", 0
            return rel, ("ok" if integrity_sha256(p) == digest else "sha_mismatch"), size

        with ThreadPoolExecutor(threads) as ex:
            for rel, verdict, n in ex.map(check, sorted(files, key=lambda r: -files[r][0])):
                res["checked"] += 1
                res["bytes"] += n
                if verdict != "ok":
                    res[verdict].append(rel)
                if res["checked"] % 10 == 0:
                    res["minutes"] = round((time.time() - t0) / 60, 1)
                    integrity_save(state)
        res["minutes"] = round((time.time() - t0) / 60, 1)
        state["done"] = True
    except Exception as exc:
        state["error"] = f"{type(exc).__name__}: {exc}"
    integrity_save(state)


if not RERUN and MODE != "fallback":
    _integrity_state = {"started_min": round((time.time() - T0) / 60, 1), "model_dir": MODEL_SRC}
    integrity_save(_integrity_state)
    threading.Thread(target=integrity_worker, args=(_integrity_state, MODEL_SRC or ""), daemon=True).start()
    print(f"input integrity check started in the background ({len(INTEGRITY['model'])} model files; hashed after the "
          "server is up)", flush=True)

In [ ]:
# Manh's cell 1 (nguyennguyen599/arc2-2026-harness-deepseek as pulled; vllm wheel: the newest in his vllm-wheels dataset; B32-M: g5e1bccd07c), verbatim
_B32M_T = globals().get("_B32M_T", {}); _B32M_T["manh_cell_1"] = round(time.time() - T0); print("[B32-M] Manh's cell 1 starts at +", _B32M_T["manh_cell_1"], "s", flush=True)
import subprocess

proc = subprocess.Popen(
    """
    sudo rm -rf /usr/local/cuda-12.8 ;
    sudo dpkg -i /kaggle/input/datasets/nguyennguyen599/cu13-install/cuda-keyring_1.1-1_all.deb &&
    cd /kaggle/input/datasets/nguyennguyen599/cu13-install &&
    sudo dpkg -i ./*.deb
    """,
    shell=True,
)

print("CUDA installation started.")

In [ ]:
# Manh's cell 2 (nguyennguyen599/arc2-2026-harness-deepseek as pulled; vllm wheel: the newest in his vllm-wheels dataset; B32-M: g5e1bccd07c), verbatim except `!rm -r /kaggle/working/*` (it would delete our fallback submission and smoke logs)
_B32M_T = globals().get("_B32M_T", {}); _B32M_T["manh_cell_2"] = round(time.time() - T0); print("[B32-M] Manh's cell 2 starts at +", _B32M_T["manh_cell_2"], "s", flush=True)
!cp /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/* /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/patches /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/third_party /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/pipeline_runtime_bootstrap /kaggle/working/
!cp -r /kaggle/input/datasets/nguyennguyen599/arc2-2026-src/vllm_delta_mhar_plugin/vllm_delta_mhar_plugin /kaggle/working/

In [ ]:
# Manh's cell 3 (nguyennguyen599/arc2-2026-harness-deepseek as pulled; vllm wheel: the newest in his vllm-wheels dataset; B32-M: g5e1bccd07c), verbatim
_B32M_T = globals().get("_B32M_T", {}); _B32M_T["manh_cell_3"] = round(time.time() - T0); print("[B32-M] Manh's cell 3 starts at +", _B32M_T["manh_cell_3"], "s", flush=True)
from concurrent.futures import ThreadPoolExecutor
import glob
import os
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys
import subprocess

%env UV_OFFLINE=1
if os.path.isdir("/kaggle/input/vllm-wheels/wheels/wheels"):
    WHEELS = Path("/kaggle/input/vllm-wheels/wheels/wheels")
else:
    WHEELS = Path("/kaggle/input/datasets/nguyennguyen599/vllm-wheels/wheels/wheels")

WORKING = Path("/kaggle/working")

os.environ["UV_FIND_LINKS"] = str(WHEELS)
os.environ["UV_OFFLINE"] = "1"

WHEELS = Path("/dev/shm/wheels")

src_dir = os.environ["UV_FIND_LINKS"]
dst_dir = "/dev/shm/wheels"
os.makedirs(dst_dir, exist_ok=True)

items_to_copy = os.listdir(src_dir)

def copy_item(item_name):
    src_path = os.path.join(src_dir, item_name)
    dst_path = os.path.join(dst_dir, item_name)
    try:
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            return f"Done copy folder: {item_name}"
        else:
            shutil.copy2(src_path, dst_path)
            return f"Done copy file: {item_name}"
    except Exception as e:
        return f"Error copy {item_name}: {e}"

print(f"Start copy {len(items_to_copy)} folder from {src_dir} to {dst_dir}...")

with ThreadPoolExecutor() as executor:
    results = executor.map(copy_item, items_to_copy)

for result in results:
    print(result)
    
print("done all")
os.environ["UV_OFFLINE"] = "1"

os.environ["UV_FIND_LINKS"] = "/dev/shm/wheels"
%env UV_FIND_LINKS=/dev/shm/wheels

In [ ]:
# Manh's cell 4 (nguyennguyen599/arc2-2026-harness-deepseek as pulled; vllm wheel: the newest in his vllm-wheels dataset; B32-M: g5e1bccd07c), verbatim except: his CUDA-install raise only in commit runs
_B32M_T = globals().get("_B32M_T", {}); _B32M_T["manh_cell_4"] = round(time.time() - T0); print("[B32-M] Manh's cell 4 starts at +", _B32M_T["manh_cell_4"], "s", flush=True)
import os
import glob
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys
import subprocess

# Supports both exact paths and wildcard patterns.
copy_patterns = []

sources = []
for pattern in copy_patterns:
    matches = glob.glob(pattern)

    if not matches:
        print(f"WARNING: no match: {pattern}")
    else:
        sources.extend(map(Path, matches))

def copy_item(src: Path):
    dst = WORKING / src.name

    if src.is_dir():
        shutil.copytree(
            src,
            dst,
            dirs_exist_ok=True,
            copy_function=shutil.copyfile,  # Faster: does not copy metadata
        )
    else:
        shutil.copyfile(src, dst)

    return src, dst


# A small worker count is usually fastest on Kaggle storage.
with ThreadPoolExecutor(max_workers=6) as executor:
    futures = [executor.submit(copy_item, src) for src in sources]

    for future in as_completed(futures):
        try:
            src, dst = future.result()
            print(f"Copied: {src} -> {dst}")
        except Exception as error:
            print(f"Copy failed: {error}")

!uv pip install --no-build-isolation vllm torch torchvision -U
!uv pip install torchaudio torchcodec -U
!uv pip install mamba-ssm==2.3.2.post1 --no-deps
!uv pip install --no-build-isolation /kaggle/working/unsloth-zoo /kaggle/working/unsloth /kaggle/working/peft
!uv pip install flash_attn causal_conv1d bitsandbytes xformers
!uv pip install ${UV_FIND_LINKS}/transformer_engine-*
!uv pip install apex redis
!uv pip install --no-build-isolation \
    flashinfer-python flashinfer-cubin flashinfer-jit-cache

!uv pip install --no-build-isolation /kaggle/working/quack-kernels

!uv pip install transformers==5.20.2

!uv pip install --no-build-isolation --no-deps \
    nvidia-cudnn-cu13==9.24.0.43 nvidia-nvshmem-cu13==3.6.5 nvidia-nccl-cu13==2.30.7

# !uv pip install nvidia-cutlass-dsl~=4.8.0 nvidia-cutlass-dsl-libs-cu13==4.8.0
!uv pip install nvidia-cutlass-dsl~=4.8.1 nvidia-cutlass-dsl-libs-cu13==4.8.1

proc.wait()

if proc.returncode != 0:
    if not RERUN:  # B32-M: a rerun keeps the fallback submission instead of erroring
        raise RuntimeError("CUDA installation failed")
    print("CUDA installation failed (rerun: continuing; the fallback stands)", flush=True)

print("CUDA installation finished.")

In [ ]:
# Manh's cell 5 (nguyennguyen599/arc2-2026-harness-deepseek as pulled; vllm wheel: the newest in his vllm-wheels dataset; B32-M: g5e1bccd07c), verbatim
_B32M_T = globals().get("_B32M_T", {}); _B32M_T["manh_cell_5"] = round(time.time() - T0); print("[B32-M] Manh's cell 5 starts at +", _B32M_T["manh_cell_5"], "s", flush=True)
!uv pip uninstall tensorflow jax
!rm -r /dev/shm/wheels

In [ ]:
# Cell 2 (B32-M): Manh's environment for B32's solver. His cells above installed CUDA 13 and his wheels into the
# kernel's Python; this replaces B32's fork wheels, overlay and venv. The solver starts his vLLM itself through a
# `vllm` wrapper (the kernel's Python -m vllm.entrypoints.cli.main, as his dsv41_vllm.py launches it). Then: an
# import check (no working vLLM -> fallback submission, as in B32), the environment record (versions, his wheel and
# dataset versions, sha256 of the files his weight loading runs: b32m_env.json) and B32's tool-python check.
# Defines what cells 2b, 3 and 5 use: V, PY, BASE_ENV, TOOL_PY, sh().
import hashlib, importlib.metadata as _md, platform, shutil
V = f"{SCRATCH}/b32m"; PY = sys.executable
MANH_VERSIONS = json.loads(r"""{"recorded_utc": "2026-10-28T18:55Z", "note": "Manh's inputs as listed through the Kaggle API at build time. kernel-metadata mounts each dataset's latest version at push time; Manh is fixing a weight-loading error (2026-10-28 ~18:45 UTC), so re-list and rebuild once he says the fix is in. The notebook records the wheel it actually mounts (b32m_env.json: vllm_wheels_in_inputs, packages, weight_loading_sha256).", "vllm_wheel": "vllm-0.32.1rc1.dev0+g5e1bccd07c-cp313-cp313-linux_x86_64.whl", "vllm_commit": "5e1bccd07c (nguyennguyen599/vllm main, 2026-10-28 16:18 UTC)", "python": "3.13 (cp313 wheels)", "key_wheels": ["torch-2.14.0+cu130-cp313", "triton-3.8.0-cp313", "humming_kernels-0.1.18", "flashinfer_python-0.7.2.post1", "flashinfer_cubin-0.7.2.post1", "flashinfer_jit_cache_sm89-0.7.2.post1+cu130", "tilelang-0.1.12"], "datasets": {"nguyennguyen599/vllm-wheels": {"version": "latest (number not listed by the API)", "files": 1669, "bytes": 9168793976, "files_created_utc": "2026-10-28 17:51-17:53"}, "nguyennguyen599/arc2-2026-src": {"version": "latest (number not listed by the API)", "files": 167, "bytes": 10399003, "files_created_utc": "2026-10-28 05:41"}, "nguyennguyen599/cu13-install": {"version": 2, "files": 107, "bytes": 4338370714, "updated_utc": "2026-10-15 16:06"}, "nguyennguyen599/unsloth-main": {"version": "latest (number not listed by the API)"}}, "docker_image": "gcr.io/kaggle-images/python@sha256:81b1e7c4b4f0f2a7e0a33943c95754b1b7af6b30d65221633a284b6b601cd84a", "weight_loading_depends_on": "only his vllm wheel (vllm/models/deepseek_v41/** incl. expert_gptq.py and common/engram.py, vllm/model_executor/model_loader/*, humming quantization + fused_humming_moe) and humming_kernels; the server is plain `python -m vllm.entrypoints.cli.main serve` (his dsv41_vllm.py applies no runtime patches and no checkpoint prep). arc2-2026-src is copied but not used by B32-M's server.", "plan_b_rowchunk": {"commit": "a6c3ee8c2b (his GitHub main; our E062 patch 01 cherry-picked)", "diff": "kaggle/dsv41_tools/b32m_patches/a6c3ee8c2b_engram_row_chunks.diff", "model_py_sha256_base_5e1bccd07c": "ead1e087c29b7d9ef76313e9849cffecfb01bec50d8625d4724527af853d10dd", "model_py_sha256_fixed_a6c3ee8c2b": "b2a04fc2a251d570f1a7bf901f55da016ee060264c2697070dc2ce3f3e614ee4", "why": "his Kaggle vllm-wheels (17:53 UTC) still holds g5e1bccd07c, which fails on the Kaggle copy: KeyError 'layers.1.engram.embed.weight.rows_0'"}}""")  # recorded by build_b32m.py (dataset versions, wheel names)


def sh(cmd, timeout=None, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, errors="replace", timeout=timeout, env=env)
    return r.returncode, (r.stdout or "") + (r.stderr or "")


os.makedirs(f"{V}/bin", exist_ok=True)
with open(f"{V}/bin/vllm", "w") as _f:  # cell 3 passes ARC_VLLM={V}/bin/vllm
    _f.write(f'#!/bin/sh\nexec "{sys.executable}" -m vllm.entrypoints.cli.main "$@"\n')
os.chmod(f"{V}/bin/vllm", 0o755)
BASE_ENV = dict(os.environ)
_cuda = next((d for d in ("/usr/local/cuda-13.0", "/usr/local/cuda") if os.path.exists(f"{d}/bin/nvcc")), None)
if _cuda:  # JIT (Triton, TileLang, Humming) needs his CUDA 13 toolchain first on PATH
    BASE_ENV.update(CUDA_HOME=_cuda, CUDA_PATH=_cuda, PATH=f"{_cuda}/bin:" + BASE_ENV.get("PATH", ""))
ENV_RECORD = {"python": sys.version.split()[0], "platform": platform.platform(), "cuda_home": _cuda,
              "manh_versions": MANH_VERSIONS, "install_cell_start_s": dict(globals().get("_B32M_T", {})),
              "env_cell_start_s": round(time.time() - T0)}
if MODE != "fallback":
    try:
        rc, out = sh(f'{PY} -c "import torch, vllm; print(torch.__version__, torch.version.cuda, '
                     f'torch.cuda.device_count(), vllm.__version__, vllm.__file__)"', timeout=900, env=BASE_ENV)
        ENV_RECORD["import_check"] = [rc, out.strip()[-400:]]
        print("vLLM import check:", rc, out.strip()[-400:], flush=True)
        assert rc == 0 and " 4 " in f" {out} ", "his vLLM does not import with 4 GPUs"
        _pk = {}
        for _p in ("vllm", "torch", "humming-kernels", "flashinfer-python", "flashinfer-cubin", "triton", "tilelang",
                   "transformers", "openai", "httpx", "tokenizers", "numpy", "scipy", "nvidia-nccl-cu13"):
            try:
                _pk[_p] = _md.version(_p)
            except Exception:
                _pk[_p] = None
        ENV_RECORD["packages"] = _pk
        ENV_RECORD["vllm_wheels_in_inputs"] = sorted(os.path.basename(p) for p in
                                                     glob.glob(f"{INPUT}/**/vllm-*.whl", recursive=True))
        # the files his weight loading of E128 runs (Kaggle Model copy): the DeepSeek-V4.1 model package (GPTQ
        # experts, Engram tables, quant config), the model loader, Humming's MoE / linear methods
        _vroot = os.path.dirname(_md.distribution("vllm").locate_file("vllm/__init__.py"))
        _files = sorted(glob.glob(f"{_vroot}/models/deepseek_v41/**/*.py", recursive=True)
                        + glob.glob(f"{_vroot}/model_executor/model_loader/*.py")
                        + glob.glob(f"{_vroot}/model_executor/layers/quantization/humming.py")
                        + glob.glob(f"{_vroot}/model_executor/layers/fused_moe/experts/fused_humming_moe.py"))
        ENV_RECORD["weight_loading_sha256"] = {os.path.relpath(f, os.path.dirname(_vroot)):
                                               hashlib.sha256(open(f, "rb").read()).hexdigest()[:16] for f in _files}
        rc, out = sh("nvcc --version | tail -1; nvidia-smi --query-gpu=driver_version,power.limit --format=csv,noheader "
                     "| head -1; free -g | head -2; df -h /tmp /kaggle/working /dev/shm | tail -3; nproc; "
                     "cat /sys/fs/cgroup/memory.max", timeout=120, env=BASE_ENV)
        ENV_RECORD["system"] = out.strip()
        print(out, flush=True)
    except Exception as exc:  # no working vLLM: keep the fallback submission
        ENV_RECORD["error"] = f"{type(exc).__name__}: {exc}"
        print("Manh's environment unusable, fallback only:", ENV_RECORD["error"], flush=True)
        MODE = "fallback"
with open(f"{WORK}/b32m_env.json", "w") as _f:
    json.dump(ENV_RECORD, _f, indent=1)
print("B32-M environment:", {k: v for k, v in ENV_RECORD.items() if k not in ("weight_loading_sha256", "system")},
      flush=True)
# The tool code runs on the image's Python (scipy etc.), as in B32.
TOOL_PY = os.environ.get("ARC_TOOL_PYTHON", sys.executable)
if MODE != "fallback":
    try:
        r = subprocess.run([TOOL_PY, "-I", "-c", "import numpy, scipy.ndimage; print(numpy.__version__)"],
                           env={"PATH": os.environ.get("PATH", ""), "OMP_NUM_THREADS": "1"}, capture_output=True,
                           text=True, timeout=300)
        print("tool python", TOOL_PY, "->", r.returncode, (r.stdout + r.stderr).strip()[-300:], flush=True)
    except Exception as exc:
        print("tool python check failed:", exc, flush=True)
BASE_ENV.pop("DSV41_SYNTH_INIT", None)
BASE_ENV["HOME"] = SUB_HOME  # /root/.cache is read-only on Kaggle with a model attached
BASE_ENV["VLLM_USE_FLASHINFER_SAMPLER"] = "0"  # as B32 and Manh's launcher
os.environ.setdefault("ARC_SMOKE_SANDBOXES", "16")  # cell 2b (commit runs only): kept small for the smoke

In [ ]:
# Cell 2m (B32-M + Manh's server): MANH'S PRODUCTION vLLM SERVER, started and owned by this cell, not by the solver.
# dsv41-vllm-prod.md's "Server only" launch: his env (MANH_SERVER_ENV, his env cell), /kaggle/working/dsv41_vllm.py
# (Manh's cell 2 copies it from arc2-2026-src) and his production flags, plus what B32's harness needs on chat turns
# (tools with tool_choice "auto": --enable-auto-tool-choice --tool-call-parser deepseek_v41) and B32's
# --prefix-cache-retention-interval 128. Waits for the model staging marker (cell 1: the FUSE mount read once), launches,
# waits until /v1/models serves dsv41, then defines MANH_BASE_URL, MANH_PORT (= ARC_PORT), MANH_MODEL, MANH_SERVER_PID,
# MANH_SERVER_LOG. Cell 3 hands MANH_BASE_URL to the solver as ARC_EXTERNAL_SERVER (S25, cell 0b's pinned solver): it
# never starts, stops or restarts a server.
# Restarts: with --supervise (arc2-2026-src's dsv41_vllm.py / qwen38_server.py have it) the server runs under his
# detached supervisor (own session; MANH_SUPERVISOR_PID): it first stops whatever holds the port (an earlier run of
# this cell included), restarts the server after an exit (OOM, crash) ARC_MAX_SERVER_RESTARTS times at most, never
# when the restart could not be up before ARC_STOP_NEW_S, and stops a start not up after ARC_SERVER_UP_TIMEOUT_S. Its
# status file MANH_SERVER_STATUS (start, state, pid, log) goes to the solver as ARC_EXTERNAL_SERVER_STATUS (S26): the
# solver waits for each restart and re-queues the attempts the death cut. Without --supervise (an older arc2-2026-src):
# one start, and its death ends new attempts (0 restarts).
# MANH_VISION=0: --language-model-only (text prompts only, more KV) instead of his vision-repl-audit recipe.
# No server: commit runs RAISE; the competition rerun keeps the fallback submission (MODE = "fallback").
import re, signal, urllib.request

# Tuned humming MoE tiles for the TP4 routed-expert shards, L4 only (num_sms 58); dsv41-vllm-prod.md, Requirements.
_moe_cfg = lambda block, warp, stages, ctas, stream_k: {
    "block_shape": block, "warp_shape": warp, "num_stages": stages, "num_ctas_per_sm": ctas,
    "use_stream_k": stream_k, "num_sms": 58, "mma_type": "mma", "output_chunk_rows": 0,
    "use_f16_accum": False, "use_pdl": False, "use_tma_a": False, "use_tma_as": False,
    "use_tma_as2": False, "use_tma_c": False}
DSV41_MOE_TUNING = {  # "NxK": [[lo, hi, config], ...], used when lo < tokens x top_k 6 <= hi
    "1152x5120": [
        [1926, 3680, _moe_cfg([32, 128, 128], [32, 64, 64], 5, 2, False)],
        [3888, 5504, _moe_cfg([48, 128, 128], [48, 64, 64], 3, 1, False)],
        [5504, 8192, _moe_cfg([64, 128, 128], [64, 32, 128], 3, 1, True)],
        [8192, 12288, _moe_cfg([48, 128, 128], [48, 64, 64], 3, 1, True)],
    ],
    "5120x576": [
        [1840, 4248, _moe_cfg([32, 256, 64], [32, 64, 64], 5, 2, False)],
        [4248, 5592, _moe_cfg([48, 256, 64], [48, 64, 32], 5, 1, False)],
        [5592, 8192, _moe_cfg([64, 256, 64], [64, 32, 64], 5, 1, False)],
    ],
}
MANH_SERVER_ENV = {  # dsv41-vllm-prod.md's env cell (os.environ.update there)
    "VLLM_HOST_ALLREDUCE": "1",
    "VLLM_HOST_ALLREDUCE_MODE": "556",            # lossless 44 + keep-warm (512), no Q8 (16): g5e1bccd07c+ wheels
    "VLLM_HOST_ALLREDUCE_PREFETCH_BLOCKS": "16",
    "VLLM_HOST_ALLREDUCE_PREFETCH_MAX_MB": "36",
    "VLLM_HOST_ALLREDUCE_LARGE_SHAPE": "32x32x256",
    "VLLM_HOST_ALLREDUCE_SHAPE": "12x12x256",
    "VLLM_HOST_ALLREDUCE_UNIT_KB": "16",
    "VLLM_EXECUTE_MODEL_TIMEOUT_SECONDS": "900",  # E066 build: safety margin (2026-10-29)
    "VLLM_TP_HOST_GATHER": "1",                   # host TP all-gather (g9065af32af wheel; older wheels ignore it)
    "VLLM_DCP_ASYNC_QUERY_GATHER": "1",           # DCP only: fp8 query all-gather on a side stream (default)
    "VLLM_HUMMING_MOE_PAD_MULTIPLE": "64,64",     # routed experts unpadded: 576-wide TP4 shard, not 640
    "VLLM_HUMMING_MOE_TUNING_CONFIG": json.dumps(DSV41_MOE_TUNING),  # 321+ token steps only
    "VLLM_TRITON_SPARSE_INDEXER_LOGITS": "1",     # indexer consumers score only layer 20's candidates
    "VLLM_DSA_DECODE_INDEXER_ROW_SHARD": "1",     # each TP rank scores 1/4 of the decode indexer rows
    "VLLM_DSV41_AR_OVERLAP_CHUNKS": "2",          # prefill all-reduces under row chunks (gf506afa210+ wheel)
    "VLLM_GPU_DRIVER_RESERVE_MB": "512",          # keep 512 MiB out of the torch cache (dsv41-vllm-prod.md, Requirements)
    "VLLM_GPU_MEMORY_LOG_INTERVAL": "60",         # per-rank "GPU memory: ..." line every 60 s
    # Prefix cache / KV layout: qwen3.8 flags, off for DSV4.1 (dsv41-vllm-prod.md, Requirements). Checkpoints from
    # the previous server ran with all four on.
    # "VLLM_KV_CACHE_GROUP_SIZE": "8",                # never read on DSV4.1's packed KV planner
    # "VLLM_PREFIX_CACHE_HOLD_REUSE_STATE": "1",    # SWA window at a running request's prompt end kept for its next turn
    # "VLLM_PREFIX_CACHE_HOLD_DECODE_STATE": "1",   # ... and at its latest decode block boundary
    # "VLLM_SWA_FINE_GRAINED_PREFIX_CACHE": "1",      # no-op for DSV4.1
}
MANH_PORT = int(os.environ.get("ARC_PORT", "18084"))
MANH_BASE_URL = f"http://127.0.0.1:{MANH_PORT}/v1"
# --context (max-model-len) and --concurrency (max-num-seqs; CUDA graphs up to it): B32-M's 163840 and 80 workers.
# dsv41-vllm-prod.md's KV and memory numbers were measured at 131072 / 64: check the KV line of the first start.
MANH_CONTEXT, MANH_SEQS = 163840, 64  # E066 build: Manh's 163840, 80 -> 163840, 64 (E066 build margins)
MANH_VISION = os.environ.get("MANH_VISION", "1") == "1"
MANH_VISION_MAX_IMAGES = int(os.environ.get("MANH_VISION_MAX_IMAGES", "16"))  # per prompt; his recipe's 16
MANH_RESTARTS = int(os.environ.get("ARC_MAX_SERVER_RESTARTS", "3"))  # cell 00: B32's 3
MANH_MODEL, MANH_SERVER_PID, MANH_SERVER_LOG = MODEL_SRC, None, None
MANH_SUPERVISOR_PID, MANH_SERVER_STATUS = None, None


def _manh_alive(pid):
    try:
        with open(f"/proc/{pid}/stat") as f:
            return f.read().rsplit(")", 1)[1].split()[0] not in ("Z", "X")
    except OSError:
        return False


def _manh_tail(n=2500):
    try:
        return " | ".join(open(MANH_SERVER_LOG, errors="replace").read().splitlines()[-15:])[-n:]
    except Exception as exc:
        return f"<no server.log: {type(exc).__name__}>"


def _manh_status():  # his supervisor's status ({} before it exists; it is replaced atomically)
    try:
        with open(MANH_SERVER_STATUS) as f:
            return json.load(f)
    except Exception:
        return {}


def _manh_logs():  # every start's server.log, oldest first (cell 4b); "" without his server
    _st, _out = _manh_status(), []
    _paths = [h.get("log") for h in _st.get("history", [])] + [_st.get("log"), MANH_SERVER_LOG]
    for _p in dict.fromkeys(p for p in _paths if p):
        try:
            _out.append(open(_p, errors="replace").read())
        except OSError:
            pass
    return "\n".join(_out)


def _manh_stop(grace=90):
    """SIGTERM his supervisor (it stops the server, then exits) or, unsupervised, the server's group; then SIGKILL both
    groups (each is a session leader: TP workers / engine core that outlived the leader). A supervisor that exited as
    failed or stopped already killed its server's group: nothing left, and those pids may belong to others by now."""
    global MANH_SERVER_PID, MANH_SUPERVISOR_PID
    _st = _manh_status()  # its pid even if the launch below never printed it; the start that runs now
    MANH_SUPERVISOR_PID = MANH_SUPERVISOR_PID or _st.get("supervisor_pid")
    MANH_SERVER_PID = _st.get("pid") or MANH_SERVER_PID
    if MANH_SUPERVISOR_PID and _st.get("state") in ("failed", "stopped") and not _manh_alive(MANH_SUPERVISOR_PID):
        return
    try:
        if MANH_SUPERVISOR_PID:
            os.kill(MANH_SUPERVISOR_PID, signal.SIGTERM)
        elif MANH_SERVER_PID:
            os.killpg(MANH_SERVER_PID, signal.SIGTERM)
    except ProcessLookupError:
        pass
    _t = time.time()
    while time.time() - _t < grace and _manh_alive(MANH_SUPERVISOR_PID or MANH_SERVER_PID):
        time.sleep(1)
    for _pid in (MANH_SUPERVISOR_PID, MANH_SERVER_PID):
        try:
            if _pid:
                os.killpg(_pid, signal.SIGKILL)
        except ProcessLookupError:
            pass


if MODE != "fallback" and MODEL_SRC:
    try:
        assert int(os.environ.get("ARC_CONTEXT_LIMIT") or MANH_CONTEXT) <= MANH_CONTEXT, "ARC_CONTEXT_LIMIT > --context"
        assert os.path.isfile(f"{WORK}/dsv41_vllm.py"), f"no {WORK}/dsv41_vllm.py (Manh's cell 2 copies arc2-2026-src)"
        if STAGE_DONE:  # the solver waited for it before loading; so does this cell
            _t = time.time()
            while not os.path.exists(STAGE_DONE) and time.time() - _t < float(os.environ.get("ARC_STAGE_WAIT_S", 2400)):
                time.sleep(5)
            _staged = open(STAGE_DONE).read().strip() if os.path.exists(STAGE_DONE) else ""
            print(f"model staging: {_staged or 'not finished'} after {time.time() - _t:.0f}s wait", flush=True)
            if _staged.startswith("ok "):
                MANH_MODEL = _staged.split(" ", 1)[1]
        _env = dict(BASE_ENV)  # cell 2's: CUDA 13 first on PATH, HOME and caches under SCRATCH
        _env.update(MANH_SERVER_ENV)
        _cudnn = "/usr/local/lib/python3.13/dist-packages/nvidia/cudnn"  # his cudnn cell
        if not os.path.isdir(_cudnn):
            import importlib.util
            _cudnn = list(importlib.util.find_spec("nvidia.cudnn").submodule_search_locations)[0]
        _env.update(CUDNN_HOME=_cudnn, CUDNN_PATH=_cudnn, LD_LIBRARY_PATH=f"{_cudnn}/lib:" + _env.get("LD_LIBRARY_PATH", ""))
        _root = f"{WORK}/dsv41_logs"
        _mode = (["--qwen38-recipe", "vision-repl-audit", "--vision-max-pixels", "2097152",
                  "--vision-max-images", str(MANH_VISION_MAX_IMAGES)] if MANH_VISION else
                 ["--qwen38-recipe", "manual", "--solve-mode", "tir", "--tir-interface", "repl", "--input-mode", "text"])
        _cmd = [PY, "-u", f"{WORK}/dsv41_vllm.py", "--qwen38-variant", "dsv4-vllm", *_mode,
                "--dsv41-vllm-model", MANH_MODEL, "--dsv41-vllm-triton-cache", f"{WORK}/dsv41_triton",
                "--output-root", _root, "--tensor-parallel-size", "4", "--gpu-ids", "0,2,1,3",
                "--qwen38-port", str(MANH_PORT), "--temp-root", "/tmp/arc_vision", "--context", str(MANH_CONTEXT),
                "--concurrency", str(MANH_SEQS), "--gpu-memory-utilization", "0.90", "--kv-offload-gib-per-rank", "10",
                "--max-num-batched-tokens", "2048",
                "--vllm-arg=--offload-backend=uva", "--vllm-arg=--cpu-offload-gb=0.35",
                "--vllm-arg=--cpu-offload-params=embed_tokens", "--vllm-arg=--engram-config.use_thp=true",
                "--vllm-arg=--watermark=0.1", "--vllm-arg=--attention-config.sparse_mla_prefill_workspace_factor=2",
                "--vllm-arg=--enable-auto-tool-choice", "--vllm-arg=--tool-call-parser=deepseek_v41",
                "--vllm-arg=--prefix-cache-retention-interval=128"]
        _up_timeout = float(os.environ.get("ARC_SERVER_UP_TIMEOUT_S", 7200))
        _stop_new = T0 + float(os.environ.get("ARC_STOP_NEW_S", 39000))
        _supervise = all("add_supervise_arguments" in open(f"{WORK}/{_f}").read()
                         for _f in ("qwen38_server.py", "dsv41_vllm.py"))
        if _supervise:
            MANH_SERVER_STATUS = f"{_root}/supervisor.json"
            _cmd += ["--supervise", str(MANH_RESTARTS), "--supervise-status", MANH_SERVER_STATUS,
                     "--supervise-deadline", str(_stop_new), "--supervise-up-timeout", str(_up_timeout)]
        else:
            print(f"WARNING: {WORK}/dsv41_vllm.py has no --supervise (an older arc2-2026-src): one start, no restart",
                  flush=True)
            MANH_RESTARTS = 0
        _r = subprocess.run(_cmd, cwd=WORK, env=_env, stdin=subprocess.DEVNULL, capture_output=True, text=True,
                            errors="replace", timeout=900)
        print("\n".join(l for l in (_r.stdout + _r.stderr).splitlines() if not l.startswith("Command:"))[-4000:],
              flush=True)  # not its "Command:" line (the full argv); supervised, that goes to supervisor.log only
        _m = re.search(r"^Supervisor PID: (\d+) ", _r.stdout, re.M)
        MANH_SUPERVISOR_PID = int(_m.group(1)) if _m else None
        assert _r.returncode == 0, f"dsv41_vllm.py exited with {_r.returncode}"
        if _supervise:
            assert MANH_SUPERVISOR_PID, "dsv41_vllm.py printed no supervisor pid"
        else:
            MANH_SERVER_PID = int(json.load(open(f"{_root}/server/launch.json"))["pid"])
            MANH_SERVER_LOG = f"{_root}/server/server.log"
        _t, _last, _models, _st = time.time(), time.time(), [], {}
        # supervised: the supervisor stops a start not up in time and gives up by itself; only ARC_STOP_NEW_S bounds it
        _deadline = _stop_new if _supervise else min(_t + _up_timeout, _stop_new)
        while True:
            if _supervise:  # the start that runs now (a restart during startup: an OOM while loading, say)
                _st = _manh_status()
                MANH_SERVER_PID, MANH_SERVER_LOG = _st.get("pid") or MANH_SERVER_PID, _st.get("log") or MANH_SERVER_LOG
                if _st.get("state") in ("failed", "stopped") or not _manh_alive(MANH_SUPERVISOR_PID):
                    raise RuntimeError(f"supervisor {_st.get('state')} (alive {_manh_alive(MANH_SUPERVISOR_PID)}): "
                                       f"{_st.get('error')}; server.log: {_manh_tail()}")
            elif not _manh_alive(MANH_SERVER_PID):
                raise RuntimeError(f"server exited during startup; server.log: {_manh_tail()}")
            if not _supervise or _st.get("state") == "up":
                try:
                    with urllib.request.urlopen(f"{MANH_BASE_URL}/models", timeout=5) as _u:
                        _models = json.loads(_u.read())["data"]
                    if any(m.get("id") == "dsv41" for m in _models):
                        break
                except Exception:
                    pass
            if time.time() > _deadline:
                raise RuntimeError(f"server not up after {time.time() - _t:.0f}s; server.log: {_manh_tail()}")
            if time.time() - _last > 120:
                _last = time.time()
                _sn = f"start {_st.get('start')} {_st.get('state')}, " if _supervise else ""
                print(f"[+{time.time() - T0:.0f}s] server loading ({_sn}{time.time() - _t:.0f}s): {_manh_tail(300)}",
                      flush=True)
            time.sleep(5)
        print(f"MANH SERVER UP after {time.time() - _t:.0f}s (+{time.time() - T0:.0f}s): {MANH_BASE_URL} model dsv41, "
              f"max_model_len {[m.get('max_model_len') for m in _models]}, pid {MANH_SERVER_PID}, log {MANH_SERVER_LOG}"
              + (f"; supervisor {MANH_SUPERVISOR_PID} (start {_st.get('start')}, {MANH_RESTARTS} restarts at most), "
                 f"status {MANH_SERVER_STATUS}" if _supervise else "; unsupervised (no restart)"), flush=True)
    except Exception as exc:
        print("=" * 100, f"MANH SERVER FAILED: {type(exc).__name__}: {exc}", "=" * 100, sep="\n", flush=True)
        try:
            _manh_stop(grace=60)
        except Exception as exc2:
            print("stopping Manh's server failed:", type(exc2).__name__, exc2, flush=True)
        MANH_SERVER_PID = MANH_SUPERVISOR_PID = MANH_SERVER_STATUS = None
        if not RERUN:
            raise
        MODE = "fallback"  # rerun: the fallback submission (cell 0) stands

In [ ]:
# Cell 2b (smoke, commit runs only): the tool sandbox at full load, before the solver starts. Starts as many sandbox
# sessions as the solver runs workers (each its own process with a 4 GiB address-space cap, 64 MB file cap, nice 10 and
# oom_score_adj 1000, like solver.session_factory), runs a typical ARC cell in every one, then checks that an oversized
# allocation fails inside its session, that a runaway cell is interrupted and the session restarts, and that closing
# every session returns host memory. Commit run: a failure RAISES ("SANDBOX CHECK FAILED"); the competition rerun skips
# this cell. Uses cell 0's KIT, MODE and RERUN, and cell 2's TOOL_PY.
import os
import sys
import threading
import time


def sandbox_meminfo_gib():
    out = {}
    with open("/proc/meminfo") as f:
        for line in f:
            key, value = line.split(":", 1)
            if key in ("MemAvailable", "AnonPages", "Shmem"):
                out[key] = int(value.split()[0]) / 2**20
    return out


def sandbox_pss_gib(pids):
    total = 0
    for pid in pids:
        try:
            with open(f"/proc/{pid}/smaps_rollup") as f:
                total += sum(int(l.split()[1]) for l in f if l.startswith("Pss:"))
        except OSError:
            pass
    return total / 2**20


def sandbox_check(kit, python, sessions, mem_mb=4096, timeout=30, alloc_gib=6):
    """Returns (report lines, failures)."""
    sys.path.insert(0, f"{kit}/repo/evals")
    from arc_harness.executor import LocalExecutor
    train = [{"input": [[0, 1, 0], [1, 1, 1]], "output": [[1, 0, 1], [0, 0, 0]]}]
    test = [[1, 0, 0], [0, 1, 1]]
    cell = ("import numpy as np\nfrom scipy import ndimage\ng = np.array(test_input)\n"
            "lab, n = ndimage.label(g)\nprint(n, (1 - g).tolist())")
    lines, failures, results, lock = [], [], [], threading.Lock()
    before = sandbox_meminfo_gib()
    exs = [LocalExecutor(train, test, timeout, 16000, memory_mb=mem_mb, python=python, style="ipython",
                         cpu_seconds=7200) for _ in range(sessions)]

    def run(ex):
        t0 = time.time()
        try:
            r = ex.execute(cell)
            ok = r.get("process_status") in (None, "ok", "completed") and "[[0, 1, 1]" in (r.get("stdout") or "")
            with lock:
                results.append((ok, time.time() - t0, (r.get("stderr") or "")[-200:]))
        except Exception as exc:
            with lock:
                results.append((False, time.time() - t0, f"{type(exc).__name__}: {exc}"))

    threads = [threading.Thread(target=run, args=(ex,)) for ex in exs]
    t0 = time.time()
    for th in threads:
        th.start()
    for th in threads:
        th.join(300)
    pids = [ex.proc.pid for ex in exs if getattr(ex, "proc", None)]
    loaded = sandbox_meminfo_gib()
    good = sum(1 for ok, _, _ in results if ok)
    slow = max((dt for _, dt, _ in results), default=0)
    lines.append(f"{good}/{sessions} sessions ran a numpy/scipy cell in {time.time() - t0:.0f} s (slowest {slow:.0f} s); "
                 f"sandbox PSS {sandbox_pss_gib(pids):.2f} GiB; host anon {before['AnonPages']:.1f} -> "
                 f"{loaded['AnonPages']:.1f} GiB, MemAvailable {before['MemAvailable']:.1f} -> {loaded['MemAvailable']:.1f} GiB")
    if good < sessions:
        failures.append(f"{sessions - good} sessions failed: {[e for ok, _, e in results if not ok][:2]}")

    # an allocation over the session's cap must fail inside the session, which stays usable
    ex = exs[0]
    r = ex.execute(f"x = bytearray({alloc_gib} * 2**30)\nprint('allocated')")
    text = (r.get("stdout") or "") + (r.get("stderr") or "")
    contained = "allocated" not in text and ("MemoryError" in text or r.get("state_reset"))
    after = ex.execute("print(6 * 7)")
    lines.append(f"{alloc_gib} GiB allocation in one session: {'blocked' if contained else 'NOT blocked'} "
                 f"({text.strip().splitlines()[-1][:90] if text.strip() else r.get('process_status')}); session after: "
                 f"{(after.get('stdout') or '').strip() or after.get('process_status')}")
    if not contained or "42" not in (after.get("stdout") or ""):
        failures.append("memory cap did not contain the allocation or the session did not recover")

    # a runaway cell is interrupted at the timeout and the session keeps working
    ex = exs[1]
    t1 = time.time()
    r = ex.execute("while True:\n    pass")
    after = ex.execute("print(6 * 7)")
    lines.append(f"runaway cell: {r.get('process_status')} after {time.time() - t1:.0f} s; session after: "
                 f"{(after.get('stdout') or '').strip() or after.get('process_status')}")
    if "42" not in (after.get("stdout") or ""):
        failures.append("session did not recover after a timeout")

    for ex in exs:
        ex.close()
    time.sleep(3)
    alive = [p for p in pids if os.path.exists(f"/proc/{p}")]
    end = sandbox_meminfo_gib()
    lines.append(f"closed: {len(alive)} sandbox processes left; host anon {end['AnonPages']:.1f} GiB, "
                 f"MemAvailable {end['MemAvailable']:.1f} GiB")
    if alive:
        failures.append(f"{len(alive)} sandbox processes outlived close()")
    if end["AnonPages"] > before["AnonPages"] + 1.0:
        failures.append(f"host anon memory not returned ({before['AnonPages']:.1f} -> {end['AnonPages']:.1f} GiB)")
    return lines, failures


if not RERUN and MODE != "fallback":
    try:
        _sb_lines, _sb_fail = sandbox_check(KIT, TOOL_PY, int(os.environ.get("ARC_SMOKE_SANDBOXES", "56")),
                                            int(os.environ.get("ARC_EXEC_MEM_MB", "4096")),
                                            int(os.environ.get("ARC_SMOKE_SANDBOX_TIMEOUT", "30")))
    except Exception as exc:
        _sb_lines, _sb_fail = [], [f"{type(exc).__name__}: {exc}"]
    for _l in _sb_lines:
        print("sandbox check:", _l, flush=True)
    if _sb_fail:
        print("=" * 100, f"SANDBOX CHECK FAILED: {_sb_fail}", "=" * 100, sep="\n", flush=True)
        raise RuntimeError(f"SANDBOX CHECK FAILED: {_sb_fail}")
    print("SANDBOX CHECK OK", flush=True)

In [ ]:
# Cell 2c (B32-M smoke, commit runs only): the API contract of the solver's server (his vLLM), checked in a
# background thread once it answers (E062 l4/api_check.py), so it costs no wall time. Required (the report cell
# fails the commit run otherwise): /v1/models serves dsv41 with max_model_len >= ARC_CONTEXT_LIMIT; /tokenize +
# /detokenize render a tool transcript; token-id /v1/completions returns token_ids (--token-exact); /metrics has
# vllm:kv_cache_usage_perc (KV gate). Recorded only: previous-turn thinking via `reasoning_content` (B32's harness
# key; upstream reads `reasoning`: only the rare full re-renders, ~0.1% of turns, lose it) and via `reasoning`, and
# a chat tool call (the harness parses tool calls itself on the token-exact path). Writes smoke_contract.json.
import re, sys, threading, urllib.request

CONTRACT_OUT = f"{WORK}/smoke_contract.json"


def contract_run(base, model="dsv41", wait_s=3600):
    res, fail = {}, []

    def post(path, body, timeout=600):
        req = urllib.request.Request(base + path, data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return json.loads(r.read())
    t = time.time()
    models = None
    while time.time() - t < wait_s:
        try:
            with urllib.request.urlopen(base + "/v1/models", timeout=5) as r:
                models = json.loads(r.read())["data"]
            break
        except Exception:
            time.sleep(10)
    res["server_up_at_s"] = round(time.time() - T0)
    if not models:
        return {**res, "fail": [f"server never answered /v1/models within {wait_s}s"], "done": True}
    res["models"] = [(m.get("id"), m.get("max_model_len")) for m in models]
    ctx = int(os.environ.get("ARC_CONTEXT_LIMIT", "163840"))
    if not any(m.get("id") == model and (m.get("max_model_len") or 0) >= ctx for m in models):
        fail.append(f"/v1/models {res['models']}: need {model} with max_model_len >= {ctx}")
    kw = {"thinking": True, "reasoning_effort": 50}
    tools = [{"type": "function", "function": {
        "name": "run_python", "description": "Run Python code in a persistent session and return stdout.",
        "parameters": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}}}]
    mark = "ZEBRA-THINKING-MARKER-42"
    try:
        for field in ("reasoning_content", "reasoning"):
            msgs = [{"role": "user", "content": "What is 2+3? Use the tool."},
                    {"role": "assistant", "content": "", field: f"I should compute it. {mark}",
                     "tool_calls": [{"id": "c1", "type": "function", "function": {
                         "name": "run_python", "arguments": json.dumps({"code": "print(2+3)"})}}]},
                    {"role": "tool", "tool_call_id": "c1", "content": "5"}]
            tk = post("/tokenize", {"model": model, "messages": msgs, "tools": tools, "chat_template_kwargs": kw,
                                    "add_generation_prompt": True})
            dt = post("/detokenize", {"model": model, "tokens": tk["tokens"]})
            res[f"thinking_via_{field}"] = mark in dt["prompt"]
            res[f"tokenize_{field}_n"] = tk.get("count")
    except Exception as exc:
        fail.append(f"/tokenize or /detokenize: {type(exc).__name__}: {exc}"[:300])
    try:
        msgs = [{"role": "system", "content": "You solve tasks with the run_python tool."},
                {"role": "user", "content": "Compute the sum of the squares of 1..10 with run_python."}]
        tk = post("/tokenize", {"model": model, "messages": msgs, "tools": tools, "chat_template_kwargs": kw,
                                "add_generation_prompt": True})
        r = post("/v1/completions", {"model": model, "prompt": tk["tokens"], "max_tokens": 64, "temperature": 0,
                                     "return_token_ids": True, "skip_special_tokens": False,
                                     "cache_salt": "b32m-contract"})
        res["completion_token_ids"] = len(r["choices"][0].get("token_ids") or [])
        if not res["completion_token_ids"]:
            fail.append("token-id /v1/completions returned no token_ids")
    except Exception as exc:
        fail.append(f"token-id /v1/completions: {type(exc).__name__}: {exc}"[:300])
    try:
        with urllib.request.urlopen(base + "/metrics", timeout=10) as r:
            m = r.read().decode(errors="replace")
        res["metrics"] = {k: bool(re.search(rf"^vllm:{k}\b", m, re.M)) for k in (
            "kv_cache_usage_perc", "num_requests_running", "num_requests_waiting", "generation_tokens_total",
            "prefix_cache_hits_total", "num_preemptions_total")}
        if not res["metrics"]["kv_cache_usage_perc"]:
            fail.append("/metrics lacks vllm:kv_cache_usage_perc (KV gate)")
    except Exception as exc:
        fail.append(f"/metrics: {type(exc).__name__}: {exc}"[:300])
    try:
        c = post("/v1/chat/completions", {"model": model, "messages": msgs, "tools": tools, "temperature": 0,
                                          "max_tokens": 4000, "chat_template_kwargs": kw})
        res["chat_tool_calls"] = len(c["choices"][0]["message"].get("tool_calls") or [])
    except Exception as exc:
        res["chat_tool_calls"] = f"{type(exc).__name__}"
    if os.environ.get("ARC_GRID_FORMAT") == "text+image":  # E066: the image harness runs on the chat path
        if not (isinstance(res["chat_tool_calls"], int) and res["chat_tool_calls"] >= 1):
            fail.append(f"chat tool calls not parsed ({res['chat_tool_calls']}): the image harness needs the "
                        "server's --tool-call-parser deepseek_v41 --enable-auto-tool-choice")
        try:  # one token-aligned grid image next to its text, as the harness sends it
            sys.path.insert(0, f"{KIT}/repo/evals")
            from arc_harness.grid_image import aligned_cell, data_uri
            g = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
            text = {"type": "text", "text": "Describe this 3x3 grid in one sentence."}
            img = {"type": "image_url", "image_url": {"url": data_uri(g, aligned_cell(g, "deepseek"))}}
            n = []
            for content in ([text], [text, img]):
                c = post("/v1/chat/completions", {"model": model, "messages": [{"role": "user", "content": content}],
                                                  "temperature": 0, "max_tokens": 16, "chat_template_kwargs": kw})
                n.append(c["usage"]["prompt_tokens"])
            res["image_prompt_tokens"] = n[1] - n[0]
            if res["image_prompt_tokens"] <= 0:
                fail.append(f"an image added no prompt tokens ({n}): the server dropped it")
        except Exception as exc:
            fail.append(f"image chat request: {type(exc).__name__}: {exc}"[:300])
    return {**res, "fail": fail, "done": True}


def contract_thread():
    try:
        out = contract_run(f"http://127.0.0.1:{os.environ.get('ARC_PORT', '8100')}")
    except Exception as exc:
        out = {"fail": [f"contract check crashed: {type(exc).__name__}: {exc}"], "done": True}
    with open(CONTRACT_OUT, "w") as f:
        json.dump(out, f, indent=1)
    print("B32-M contract:", out, flush=True)


if not RERUN and MODE != "fallback":
    _ct_thread = threading.Thread(target=contract_thread, daemon=True)  # E066: cell 3w waits for it
    _ct_thread.start()
    print("B32-M contract check started (background)", flush=True)

In [ ]:
# Cell 3: the solver (tool harness, deadlines, watchdog, incremental submission). B32-M + Manh's server: it calls
# his server (cell 2m) at ARC_EXTERNAL_SERVER = MANH_BASE_URL (S25: no launch, no stop, no restart; its log gets
# "server up after ...s (external, start K)" with the KV line of that start's server.log). S26: it follows his
# supervisor through ARC_EXTERNAL_SERVER_STATUS = MANH_SERVER_STATUS: after a death it waits for the next start
# (ARC_MAX_SERVER_RESTARTS = MANH_RESTARTS times at most) and re-queues the attempts it cut; once the supervisor
# gives up it finishes with what it has. The model dir is his server's (the tokenizer for the loop guard); cell
# 2m already waited for the staging marker.
if MODE != "fallback" and globals().get("MANH_SERVER_PID"):
  try:
    import signal, threading
    env = dict(BASE_ENV)
    env.update(ARC_KIT=KIT, ARC_MODEL_DIR=MANH_MODEL, ARC_STAGE_DONE="", ARC_EXTERNAL_SERVER=MANH_BASE_URL,
               ARC_EXTERNAL_SERVER_LOG=MANH_SERVER_LOG, ARC_EXTERNAL_SERVER_STATUS=MANH_SERVER_STATUS or "",
               ARC_MAX_SERVER_RESTARTS=str(MANH_RESTARTS), ARC_TOOL_PYTHON=TOOL_PY, ARC_SCRATCH=SCRATCH)
    if MODE == "smoke":
        env.setdefault("ARC_TASK_LIMIT", "8")
        for k, v in (("ARC_STOP_NEW_S", "4800"), ("ARC_FORCE_AT_S", "5400"), ("ARC_HARD_STOP_S", "5700")):
            env.setdefault(k, v)
    p = subprocess.Popen([PY, "-u", f"{KIT}/solver_dsv41.py"], env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, errors="replace", start_new_session=True)
    # Independent kill switch: whatever hangs inside the solver, its whole session dies at T0 + ARC_KILL_AT_S
    # (default 11h45; the solver's own hard stop is 11h30), so cell 4 and the notebook end before Kaggle's 12 h.
    def _kill():
        try:
            os.killpg(p.pid, signal.SIGKILL)
            print("kill switch: solver session killed", flush=True)
        except Exception:
            pass
    _timer = threading.Timer(max(0.0, T0 + float(os.environ.get("ARC_KILL_AT_S", 42300)) - time.time()), _kill)
    _timer.daemon = True
    _timer.start()
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    _timer.cancel()
    print("solver exit code", p.returncode, flush=True)
  except Exception as exc:
    print("solver cell failed:", type(exc).__name__, exc, flush=True)

In [ ]:
# Cell 3w (E066, commit runs only): cell 2c's API contract runs in a background thread and needs the server. The light
# smoke's 2 tasks can finish before it does, and cell 4 then stops the server, so wait for it here (at most 15 min).
import os, time as _wt
if not RERUN and "_ct_thread" in globals():
    _wt0 = _wt.time()
    _ct_thread.join(timeout=900)
    print("E066: contract thread", "finished" if not _ct_thread.is_alive() else "STILL RUNNING after 900 s",
          f"(waited {_wt.time() - _wt0:.0f} s)", flush=True)


In [ ]:
# Cell 4: final check of submission.json; log archive.
import tarfile
def _ok(g):
    return (isinstance(g, list) and 0 < len(g) <= 30 and all(isinstance(r, list) and 0 < len(r) <= 30 for r in g)
            and len({len(r) for r in g}) == 1 and all(type(v) is int and 0 <= v <= 9 for r in g for v in r))
try:
    sub = json.load(open(f"{WORK}/submission.json"))
    assert set(sub) == set(_tasks), f"task ids differ ({len(sub)} vs {len(_tasks)})"
    for t, v in _tasks.items():
        assert len(sub[t]) == len(v["test"]), f"{t}: {len(sub[t])} rows for {len(v['test'])} test inputs"
        assert all(_ok(r["attempt_1"]) and _ok(r["attempt_2"]) for r in sub[t]), f"{t}: invalid grid"
    print("submission.json valid", flush=True)
except Exception as exc:
    print("submission.json invalid, writing the fallback:", exc, flush=True)
    json.dump(_fb, open(f"{WORK}/submission.json", "w"))
try:
    logs = f"{WORK}/dsv41_logs"
    if os.path.isdir(f"{logs}/traces"):
        with tarfile.open(f"{WORK}/dsv41_traces.tar.gz", "w:gz") as tf:
            tf.add(f"{logs}/traces", arcname="traces")
        shutil.rmtree(f"{logs}/traces", ignore_errors=True)
    for f in ("stage.log",):
        if os.path.exists(f"{SCRATCH}/{f}"):
            shutil.copy(f"{SCRATCH}/{f}", f"{logs}/{f}" if os.path.isdir(logs) else f"{WORK}/{f}")
except Exception as exc:
    print("log archive failed:", exc, flush=True)
# B32-M + Manh's server: stop it (the solver leaves an external server alone); his supervisor first, so that it
# does not restart it.
if globals().get("MANH_SERVER_PID"):
    try:
        _manh_stop()
    except Exception as exc:
        print("stopping Manh's server failed:", type(exc).__name__, exc, flush=True)
    print("Manh's server stopped; supervisor:", _manh_status().get("state") or "none", flush=True)
print(f"done at +{(time.time() - T0) / 3600:.2f}h", flush=True)

In [ ]:
# Cell 4b (B32-M smoke report, commit runs only): the timeline (install cells, solver start, server start /
# up), the server's throughput windows (gen tok/s, running, KV every 30 s), the environment record (b32m_env.json) and
# the API contract (smoke_contract.json). If the server never came up, the timeline and the server log's tail say how
# far it got (install, weight load, KV/graph setup). It only prints; the verdict cell at the end raises on a failure.
import re
if not os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    _b32m_bad = []
    _w = os.environ.get("ARC_OUT", "/kaggle/working")
    _ev = {}
    try:
        _ev = json.load(open(f"{_w}/b32m_env.json"))
        print("B32-M env:", {k: _ev.get(k) for k in ("python", "packages", "vllm_wheels_in_inputs",
                                                      "error")}, flush=True)
        if _ev.get("error"):
            _b32m_bad.append(f"environment: {_ev['error']}")
    except Exception as exc:
        _b32m_bad.append(f"b32m_env.json: {type(exc).__name__}: {exc}")
    _sl = ""
    try:
        _sl = open(f"{_w}/dsv41_logs/solver.log").read()
    except Exception as exc:
        _b32m_bad.append(f"solver.log: {type(exc).__name__}: {exc}")

    def _at(pattern):  # pod-clock hours of the first solver.log line matching `pattern`, in minutes
        m = re.search(r"\+(\d+\.\d+)h\] [^\n]*" + pattern, _sl)
        return round(float(m.group(1)) * 60, 1) if m else None
    _tl = {"manh_install_cells_start_s": _ev.get("install_cell_start_s"), "env_cell_start_s": _ev.get("env_cell_start_s"),
           "solver_start_min": _at("fallback submission written"),
           "server_start_min": _at("(server start \\(first|\\(wait 0\\): started by the notebook)"),
           "server_up_min": _at("server up after"), "server_failed_min": _at("server (exited during startup|not up after)"),
           "first_attempt_done_min": _at(r"\[1\] "), "force_cut_min": _at("force deadline"),
           "final_write_min": _at("submission written \\(final\\)")}
    _up = re.findall(r"server up after (\d+)s(?: \([^)]*\))?; (.*)", _sl)
    _tl["server_up"] = _up[0] if _up else None
    print("B32-M timeline (minutes after the first cell; install cells in seconds):", _tl, flush=True)
    if globals().get("MANH_SERVER_STATUS"):  # his supervisor: every start of his server, and how it ended
        _ss = _manh_status()
        print("Manh's supervisor:", {k: _ss.get(k) for k in ("state", "start", "max_restarts", "error")}, "| starts:",
              [{k: h.get(k) for k in ("start", "up_s", "ran_s", "error")} for h in _ss.get("history", [])], flush=True)
    if not _up:
        _b32m_bad.append("server never came up (timeline above)")
        try:  # how far the server got: weight load, KV pool, CUDA graphs
            _sv = _manh_logs() or open(globals().get("MANH_SERVER_LOG") or f"{_w}/dsv41_logs/server.log", errors="replace").read()
            for _pat in ("Loading weights took", "Model loading took", "GPU KV cache size", "Capturing CUDA graph",
                         "Graph capturing finished", "Starting vLLM API server", "Error", "Traceback", "KeyError"):
                _hits = [l for l in _sv.splitlines() if _pat in l]
                if _hits:
                    print(f"  server.log [{_pat}] x{len(_hits)}: {_hits[-1][-240:]}", flush=True)
            print("  server.log tail:", " | ".join(_sv.splitlines()[-12:])[-2500:], flush=True)
        except Exception as exc:
            print("  no server.log:", type(exc).__name__, exc, flush=True)
    else:
        print("B32-M attempts done:", _sl.count(" stop="), flush=True)
    try:
        _mt = [json.loads(l) for l in open(f"{_w}/dsv41_logs/metrics.jsonl") if l.strip()]
        print("B32-M server windows (min, gen tok/s, server running, waiting, KV, prefix hit):",
              [(round(m.get("elapsed_h", 0) * 60, 1), m.get("gen_tok_s"), m.get("srv_running"), m.get("srv_waiting"),
                m.get("kv"), m.get("prefix_hit")) for m in _mt], flush=True)
        _gt = [m["gen_tok_s"] for m in _mt if m.get("gen_tok_s")]
        print("B32-M gen tok/s: max", max(_gt, default=None), "| mean of windows", round(sum(_gt) / len(_gt), 1)
              if _gt else None, "| KV max", max((m.get("kv") or 0 for m in _mt), default=None), flush=True)
    except Exception as exc:
        print("B32-M metrics unavailable:", type(exc).__name__, exc, flush=True)
    if "submission written (final)" not in _sl:
        _b32m_bad.append("no final submission write")
    try:
        import time as _ct
        for _ in range(6):  # the contract thread writes its file once the server answered
            if os.path.exists(f"{_w}/smoke_contract.json"):
                break
            _ct.sleep(5)
        _ck = json.load(open(f"{_w}/smoke_contract.json"))
        print("B32-M contract:", _ck, flush=True)
        _b32m_bad += [f"contract: {x}" for x in _ck.get("fail", [])]
    except Exception as exc:
        _b32m_bad.append(f"smoke_contract.json: {type(exc).__name__}: {exc}")
    try:  # E066: the restart supervisor's history (cell 2s); informational
        print("E066 supervisor status:", open(f"{_w}/dsv41_logs/e066_supervisor.json").read().strip(),
              "| events:", (globals().get("SUPERVISOR") or {}).get("events"), flush=True)
    except Exception as exc:
        print("E066 supervisor status: none", type(exc).__name__, flush=True)
    try:  # E066: cell 1b's names-and-sizes pass over every model file
        _ms = json.load(open(f"{_w}/smoke_integrity.json")).get("model_stat")
        print("E066 model files (names, sizes):", _ms, flush=True)
        if not _ms or _ms["missing"] or _ms["size_mismatch"]:
            _b32m_bad.append(f"environment: model files {_ms}")
    except Exception as exc:
        _b32m_bad.append(f"environment: smoke_integrity.json: {type(exc).__name__}: {exc}")
    if os.environ.get("B32M_LOAD_TEST") == "1":  # load test: throughput is the point, finished tasks are not
        try:  # vLLM's 10-s stats lines: pure-decode windows = no prompt tokens and >= 70 running
            _sv = _manh_logs() or open(globals().get("MANH_SERVER_LOG") or f"{_w}/dsv41_logs/server.log", errors="replace").read()
            _wins = [(float(a), float(b), int(c)) for a, b, c in re.findall(
                r"Avg prompt throughput: ([\d.]+) tokens/s, Avg generation throughput: ([\d.]+) tokens/s, Running: (\d+) reqs",
                _sv)]
            _pure = sorted(g for pr, g, r in _wins if pr == 0 and r >= 70)
            _q = lambda xs, f: xs[min(len(xs) - 1, int(f * len(xs)))] if xs else None
            _act = [g for pr, g, r in _wins if r > 0]
            print("B32-M LOAD pure-decode windows (prompt 0 tok/s, running >= 70):", {"n": len(_pure), "median": _q(_pure, .5),
                  "p90": _q(_pure, .9), "max": _pure[-1] if _pure else None}, flush=True)
            print("B32-M LOAD server windows (10 s):", {"n": len(_wins), "mean gen tok/s while running": round(
                sum(_act) / len(_act), 1) if _act else None, "running max": max((r for _, _, r in _wins), default=None),
                "windows with prompt tokens": sum(1 for pr, _, _ in _wins if pr > 0)}, flush=True)
            _m30 = [m.get("gen_tok_s") for m in _mt if m.get("gen_tok_s") is not None] if "_mt" in dir() else []
            print("B32-M LOAD solver windows (30 s) gen tok/s:", _m30, "| mean", round(sum(_m30) / len(_m30), 1)
                  if _m30 else None, flush=True)
        except Exception as exc:
            print("B32-M LOAD stats unavailable:", type(exc).__name__, exc, flush=True)
        _crash = [x for x in ("server died", "server exited during startup", "solver crashed", "server not up after")
                  if x in _sl]
        _fatal = [x for x in _b32m_bad if x.startswith(("server never", "environment")
                  + (("contract",) if os.environ.get("ARC_GRID_FORMAT") == "text+image" else ()))]
        _fatal += [f"solver.log: {x}" for x in _crash]
        try:
            _sub = json.load(open(f"{_w}/submission.json"))
            assert isinstance(_sub, dict) and _sub, "empty"
        except Exception as exc:
            _fatal.append(f"no valid submission.json ({type(exc).__name__}: {exc})")
        for _x in _b32m_bad:
            if _x not in _fatal:
                print("B32-M LOAD warning (not fatal in a load test):", _x, flush=True)
        _b32m_bad = _fatal
    if os.environ.get("E066_LIGHT_SMOKE") == "1":  # E066 light smoke: no crash, real answers, a valid submission
        _b32m_bad += [f"solver.log: {x}" for x in ("server died", "server exited during startup", "solver crashed",
                                                  "server not up after") if x in _sl]
        try:
            _att = [json.loads(x) for x in open(f"{_w}/dsv41_logs/attempts.jsonl") if x.strip()]
            _ans = [x for x in _att if x.get("pred") and not x.get("error")]
            print(f"E066 light smoke: {len(_att)} attempt records, {len(_ans)} with an answer,",
                  sum(bool(x.get("correct")) for x in _att), "correct", flush=True)
            if not _ans:
                _b32m_bad.append(f"no attempt produced an answer ({len(_att)} attempt records)")
        except Exception as exc:
            _b32m_bad.append(f"attempts.jsonl: {type(exc).__name__}: {exc}")
        try:
            _sub = json.load(open(f"{_w}/submission.json"))
            assert isinstance(_sub, dict) and _sub, "empty"
        except Exception as exc:
            _b32m_bad.append(f"no valid submission.json ({type(exc).__name__}: {exc})")
        try:  # E066: host RAM headroom under Kaggle's cgroup, projected from this smoke to a full load. Pod 2,
            # 2026-10-29: the engine grows ~4 GiB from server up to load; solver + sandboxes + kernel ~0.2 GiB/worker
            _env = [json.loads(x) for x in open(os.environ.get("ARC_SMOKE_ENV_LOG", f"{_w}/smoke_env.jsonl")) if x.strip()]
            _st = [x["stat"] for x in _env if isinstance(x.get("stat"), dict) and "anon" in x["stat"]]
            _peak = max((s_["anon"] + s_.get("shmem", 0)) / 2**30 for s_ in _st)
            _lim = min(float(x["memory.max"]) for x in _env if str(x.get("memory.max", "")).replace(".", "").isdigit()) / 2**30
            _nw = int(os.environ.get("ARC_WORKERS", "64"))
            _proj = _peak + 4 + 0.2 * max(0, _nw - 2)
            print(f"E066 host RAM: smoke peak anon+shmem {_peak:.1f} GiB; projected at {_nw} workers {_proj:.1f} GiB; "
                  f"cgroup limit {_lim:.1f} GiB; headroom {_lim - _proj:.1f} GiB", flush=True)
            if _lim - _proj < 5:
                _b32m_bad.append(f"host RAM: projected {_proj:.1f} GiB at {_nw} workers leaves {_lim - _proj:.1f} GiB "
                                 f"under the {_lim:.1f} GiB cgroup (need >= 5)")
            elif _lim - _proj < 15:
                print("E066 host RAM WARNING: headroom under 15 GiB", flush=True)
        except Exception as exc:
            _b32m_bad.append(f"host RAM check: {type(exc).__name__}: {exc}")
    B32M_BAD = _b32m_bad  # the verdict cell at the end raises on it (after B32's smoke report has printed)

In [ ]:
# Cell 5 (smoke report, commit runs only): peaks of host memory (cgroup, the 173 GiB limit), GPU power / clocks /
# memory / temperature and the lowest free disk, from the sampler started in the settings cell (every 30 s)
import json, os
if not os.environ.get("KAGGLE_IS_COMPETITION_RERUN") and os.environ.get("ARC_SMOKE_ENV_LOG"):
    try:
        _recs = [json.loads(l) for l in open(os.environ["ARC_SMOKE_ENV_LOG"]) if l.strip()]
        _gib = lambda v: round(int(v) / 2**30, 1) if v not in (None, "", "max") and str(v).isdigit() else v
        _mx = lambda xs: max([x for x in xs if isinstance(x, (int, float))], default=None)
        print(f"smoke report: {len(_recs)} samples over {(_recs[-1]['t'] - _recs[0]['t']) / 60:.1f} min", flush=True)
        print("  cgroup memory.max", _gib(_recs[-1].get("memory.max")), "GiB; peak memory.current",
              _mx([_gib(r.get("memory.current")) for r in _recs]), "GiB; memory.peak", _gib(_recs[-1].get("memory.peak")),
              "GiB; swap", _gib(_recs[-1].get("memory.swap.current")), flush=True)
        for k in ("anon", "shmem", "file", "unevictable"):
            print(f"  peak cgroup {k}", _mx([_gib(r.get("stat", {}).get(k)) for r in _recs]), "GiB", flush=True)
        _mi = [r.get("meminfo_kb", {}) for r in _recs]
        print("  host MemTotal", round(_mi[-1].get("MemTotal", 0) / 2**20, 1), "GiB; min MemAvailable",
              round(min(m.get("MemAvailable", 0) for m in _mi) / 2**20, 1), "GiB; peak Shmem",
              round(max(m.get("Shmem", 0) for m in _mi) / 2**20, 1), "GiB; peak Mlocked",
              round(max(m.get("Mlocked", 0) for m in _mi) / 2**20, 1), "GiB", flush=True)
        _g = {}
        for r in _recs:
            for line in r.get("gpu", []):
                f = [x.strip() for x in line.split(",")]
                if len(f) == 7:
                    d = _g.setdefault(f[0], {"draw": [], "limit": f[2], "sm": [], "mem": [], "total": f[5], "temp": []})
                    for k, i in (("draw", 1), ("sm", 3), ("mem", 4), ("temp", 6)):
                        try:
                            d[k].append(float(f[i]))
                        except ValueError:
                            pass
        for i, d in sorted(_g.items()):
            print(f"  GPU {i}: power limit {d['limit']} W, peak draw {_mx(d['draw'])} W, peak SM {_mx(d['sm'])} MHz, "
                  f"peak memory {_mx(d['mem'])} / {d['total']} MiB, peak {_mx(d['temp'])} C", flush=True)
        _df = {}
        for r in _recs:
            for k, v in r.get("disk_free_gib", {}).items():
                _df[k] = min(_df.get(k, v), v)
        print("  lowest free disk (GiB):", _df, flush=True)
        _sb = [r.get("sandbox", {}) for r in _recs]
        print("  tool sandboxes: peak", max((s.get("n", 0) for s in _sb), default=0), "processes,",
              max((s.get("rss_gib", 0) for s in _sb), default=0), "GiB RSS", flush=True)
    except Exception as exc:
        print("smoke report failed:", type(exc).__name__, exc, flush=True)
    try:  # which code paths the smoke exercised (solver log + server metrics)
        _lg = f"{os.environ.get('ARC_OUT', '/kaggle/working')}/dsv41_logs"
        _m = [json.loads(l) for l in open(f"{_lg}/metrics.jsonl") if l.strip()]
        _s = open(f"{_lg}/solver.log").read()
        _host = sum((x.get("host_tok_s") or 0) * (x.get("window_s") or 0) for x in _m)
        print("smoke coverage:", {"KV max": max((x.get("kv") or 0) for x in _m), "server running max":
              max((x.get("srv_running") or 0) for x in _m), "preemptions": max((x.get("preemptions") or 0) for x in _m),
              "tokens restored from host (KV offload loads)": int(_host), "KV gate holds logged": _s.count("KV gate:"),
              "server restarts": _s.count("server start (restart"), "attempts done": _s.count(" stop="),
              "force cut": "force deadline:" in _s, "final write": "submission written (final)" in _s}, flush=True)
    except Exception as exc:
        print("smoke coverage failed:", type(exc).__name__, exc, flush=True)
    _integrity_bad = None
    _hp = f"{os.environ.get('ARC_OUT', '/kaggle/working')}/smoke_hang.json"
    if os.path.exists(_hp):  # cell 2c: a deliberate engine hang; the solver's watchdog must restart the server
        _hg = json.load(open(_hp))
        try:
            _hs = open(f"{os.environ.get('ARC_OUT', '/kaggle/working')}/dsv41_logs/solver.log").read()
        except OSError:
            _hs = ""
        _after = _hs.split("server stalled", 1)[1] if "server stalled" in _hs else ""
        print("hang test:", _hg, "| watchdog fired:", "server stalled" in _hs, "| restarts:", _hs.count("server start (restart"),
              "| re-queued after the stall:", _after.count("re-queue "), "| attempts finished after the stall:",
              _after.count(" stop="), "| conn retries spent on a dead server:", _after.count("(2):") + _after.count("(3):"),
              flush=True)
        if _hg.get("error") or _hg.get("resumed_without_restart") or "server stalled" not in _hs:
            _integrity_bad = "HANG TEST FAILED: the watchdog did not restart the frozen server (see the line above)"
        else:
            print("HANG TEST OK: frozen engine detected and the server restarted", flush=True)
    _ip = f"{os.environ.get('ARC_OUT', '/kaggle/working')}/smoke_integrity.json"
    if os.path.exists(_ip):  # cell 1b's background check of every input (model hashed after the server came up)
        import time as _it
        for _ in range(18):  # B32-M: wait up to 90 s for the model hash (<= 50-min smoke)
            _ig = json.load(open(_ip))
            if _ig.get("done") or _ig.get("error"):
                break
            _it.sleep(5)
        _md = _ig.get("model", {})
        print("input integrity: competition files", _ig.get("competition"), "| kit:", _ig.get("kit"), "| wheels:",
              _ig.get("wheels"), "| model:", {k: _md.get(k) for k in ("files", "checked", "missing", "size_mismatch",
              "sha_mismatch", "started_min", "minutes")}, "| error:", _ig.get("error"), flush=True)
        _mbad = any(_md.get(k) for k in ("missing", "size_mismatch", "sha_mismatch"))
        if not _ig.get("done") and not (_ig.get("error") or _mbad):  # B32-M: an unfinished hash is reported
            print("INPUT INTEGRITY INCOMPLETE (short smoke): no mismatch among the files hashed so far", flush=True)
        elif _ig.get("error") or _mbad or (_ig.get("wheels") or {}).get("bad"):
            _integrity_bad = "INPUT INTEGRITY FAILED or incomplete: see the line above"
        else:
            print("INPUT INTEGRITY OK: every model file matches its sha256", flush=True)
    if _integrity_bad:
        print("=" * 100, _integrity_bad, "=" * 100, sep=chr(10), flush=True)
        raise RuntimeError(_integrity_bad)

In [ ]:
# B32-M smoke verdict (commit runs only): RAISES on any failure the B32-M report cell found, so the commit run errors.
import os
if not os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    _bad = globals().get("B32M_BAD", ["the B32-M report cell did not run"])
    if _bad:
        print("=" * 100, f"B32-M SMOKE FAILED: {_bad}", "=" * 100, sep="\n", flush=True)
        raise RuntimeError(f"B32-M SMOKE FAILED: {_bad}")
    print("B32-M SMOKE OK", flush=True)